# AhToVIK validation suite (Google Colab)

Compares **knowledge-driven AhToVIK** (decision logic + evidence gate + knowledge loop) with optimisation and related-work baselines, on the factors of the evaluation table and on resilience.

| Part | What is measured | Methods |
|---|---|---|
| **1 Static decision benchmark** | latency (build / solve / total), scalability (log-log slope, N = 10 ... 10,000), optimality gap vs the certified CO ceiling, fill-rate and priority-weighted unmet demand under scarcity (0.8x ... 10x), robustness (rank stability over v, solution stability over BIG_M), determinism / solution multiplicity, implementation complexity | AHTOVIK (+ no-gate and partial-staffing variants), CO-MILP (BIG_M), SAW / TOPSIS / VIKOR greedy [15][16], trait-coverage matching [5][14], sequential auction, random |
| **2 Dynamic resilience benchmark** | mission continuity (on-time completion, progress at deadline, makespan), recovery latency, downtime, unnecessary and repeated ineffective interventions, dual-authority and duplicate dispatches, work lost, escalations, decision overhead; resilience curves vs fault count, noise, packet loss, fleet size | AHTOVIK, AHTOVIK-frozen, AHTOVIK-nogate, MARKET [13], RAINBOW (ReBeT/SUAVE/Stitch-style) [7][8][17][19], MAYYA [5], CALVO [6], FILIPPONE [12], ESFAHANI [3], CARWEHL [10], CAMARA [11], RODRIGUES [20] |
| **3 Knowledge-loop benchmark** | does historical knowledge speed up later recoveries, and what does it cost when the prior is wrong | recurring / drift / spurious / non-recurring mission streams |

**Run it:** `Runtime -> Run all`. `SMOKE = False` (first cell) uses **500 independent paired runs per case** (`N_RUNS`, `N_DYN`, `N_SEQ`); the scalability sweep follows the table (500 runs up to N=100, fewer for larger N; see `SCALE_RUNS`). Set `SMOKE = True` for a ~5-minute end-to-end check. Every case is cached in `results/cache`, so after a disconnect simply re-run. All tables, figures, CSV and LaTeX land in `results/` and are zipped at the end.

**Read before quoting numbers.**
1. The dynamic world encodes the mechanisms of the paper (zone/mode restrictions with ambiguous symptoms, expiring authority, checkpoints, diagnostics). Baselines are compact re-implementations of each approach's mechanism, not the original code, and run on the identical plant, telemetry and handover substrate.
2. Design choices that the paper text must mirror: closed-world two-cause diagnostic model (zone/mode restriction vs isolated robot fault), escalation cooldown (operator takes over; automatic retries stop), complete-team-only staffing, lease length and clock margin, and the knowledge-reuse rule (order diagnostics by expected time given recorded cause frequencies).
3. The static baselines are the *allocation cores* of the dynamic ones: CO = Mayya/Calvo/Filippone global assignment, SSI/greedy = Market/Rainbow/Esfahani/Carwehl/Camara/Rodrigues initial allocation.

In [ ]:
import numpy as np, pandas as pd, math, time, json, itertools, warnings, os, sys, inspect, ast, textwrap, hashlib, pickle
from dataclasses import dataclass, field, replace
from collections import defaultdict, deque
import matplotlib.pyplot as plt
from scipy import stats, sparse
from scipy.optimize import milp, LinearConstraint, Bounds, linear_sum_assignment

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

# =============================================================================
# CONFIGURATION
#   N_RUNS = independent runs per evaluation case (seeded, paired across methods).  Spec: 500.
#   SMOKE  = True shrinks everything (10 runs / small N grid) to check the notebook end-to-end in minutes.
#   Results are cached per case in ./results/cache so an interrupted Colab session can simply be re-run.
# =============================================================================
SMOKE = False
N_RUNS = 12 if SMOKE else 500
# scalability sweep follows the evaluation table (N grid, 15-20 seeds); small N get more seeds for tighter CIs
SCALE_N = [10, 30, 100, 300, 1000, 3000, 10000] if not SMOKE else [10, 30, 100, 300, 1000]
SCALE_RUNS = ({10: 500, 30: 500, 100: 500, 300: 100, 1000: 40, 3000: 20, 10000: 15} if not SMOKE
              else {10: 6, 30: 6, 100: 6, 300: 4, 1000: 3})
N_DYN = 12 if SMOKE else 500          # runs per (scenario, method) in the dynamic resilience benchmark
N_SEQ = 8 if SMOKE else 500           # independent episode sequences for the knowledge-loop (recurrence) benchmark
N_WORKERS = max(1, (os.cpu_count() or 1))
MASTER_SEED = 20261022
OUT = "results"; CACHE = f"{OUT}/cache"
os.makedirs(CACHE, exist_ok=True)

def cached(name, fn, force=False):
    """Run fn() once per (name, SMOKE/N settings) and cache the resulting DataFrame on disk (Colab-disconnect safe)."""
    key = hashlib.md5(f"{name}|{SMOKE}|{N_RUNS}|{N_DYN}|{N_SEQ}".encode()).hexdigest()[:8]
    path = f"{CACHE}/{name}_{key}.pkl"
    if os.path.exists(path) and not force:
        print(f"[cache] {name}: loaded {path}"); return pd.read_pickle(path)
    t0 = time.time(); df = fn(); df.to_pickle(path)
    print(f"[run] {name}: {len(df)} rows in {time.time() - t0:.0f}s"); return df

def pmap(fn, items, chunksize=8):
    """Parallel map over picklable items (fork-based; falls back to serial)."""
    if N_WORKERS <= 1 or len(items) < 4 * N_WORKERS:
        return [fn(x) for x in items]
    import multiprocessing as mp
    try:
        ctx = mp.get_context("fork")
        with ctx.Pool(N_WORKERS) as pool:
            return pool.map(fn, items, chunksize=chunksize)
    except Exception as e:
        print("parallel map failed, running serially:", type(e).__name__); return [fn(x) for x in items]

print(f"SMOKE={SMOKE} | N_RUNS={N_RUNS} | N_DYN={N_DYN} | N_SEQ={N_SEQ} | workers={N_WORKERS} | scale grid={SCALE_N}")

## A. Mission model: heterogeneous fleet, criteria, AHP profiles (CR < 0.1), instance generator

In [ ]:
# ============================================================================
# A. MISSION MODEL (shared by the static and the dynamic benchmark)
#    Criteria follow the paper: navigation efficiency, payload, safety, communication, adaptability, energy (cost).
# ============================================================================
CRIT = ["nav", "payload", "safety", "comm", "adapt", "energy"]
BEN = np.array([1, 1, 1, 1, 1, 0], bool)                 # energy is a cost criterion
TYPES = ["UAV", "UGV", "CREEPER"]
# nominal (lo, hi) range of each criterion per robot type -> heterogeneity in scale (payload kg vs. unit-less scores)
TYPE_SPEC = {
    "UAV":     [(0.70, 0.95), (0.5, 2.0),   (0.55, 0.85), (0.65, 0.95), (0.50, 0.90), (0.60, 0.90)],
    "UGV":     [(0.40, 0.80), (20.0, 60.0), (0.60, 0.90), (0.50, 0.85), (0.40, 0.80), (0.30, 0.60)],
    "CREEPER": [(0.20, 0.55), (5.0, 15.0),  (0.50, 0.80), (0.35, 0.70), (0.60, 0.95), (0.30, 0.50)],
}
TYPE_FRAC = {"UAV": 0.25, "UGV": 0.50, "CREEPER": 0.25}
GROUND = ("UGV", "CREEPER")

# ---- AHP: context-specific criteria + approved preference profile, with the consistency circuit breaker (CR < 0.1)
SAATY = np.array([1 / k for k in range(9, 1, -1)] + [1.0] + list(range(2, 10)), dtype=float)
RI = {1: 0.0, 2: 0.0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32}

def pairwise_from_priorities(w):
    n = len(w); M = np.ones((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            v = SAATY[np.argmin(np.abs(np.log(SAATY) - np.log(w[i] / w[j])))]
            M[i, j], M[j, i] = v, 1.0 / v
    return M

def ahp_weights(M):
    vals, vecs = np.linalg.eig(M)
    k = int(np.argmax(vals.real)); lam = vals[k].real
    w = np.abs(vecs[:, k].real); w /= w.sum()
    n = len(M); ci = (lam - n) / (n - 1) if n > 1 else 0.0
    cr = ci / RI[n] if RI.get(n, 0) > 0 else 0.0
    return w, cr

# role -> eligible types, selected criteria (+ 'health' is appended at run time for the dynamic benchmark),
#         approved preference profile, hard capability requirements (eligibility filter)
ROLES = {
    "scout":   dict(types=("UAV",),           crit=["nav", "comm", "safety", "energy"], prio=[.35, .25, .20, .20], req={"comm": 0.70}),
    "rescuer": dict(types=("UGV",),           crit=["payload", "nav", "safety", "energy"], prio=[.35, .20, .25, .20], req={"payload": 28.0}),
    "carrier": dict(types=("UGV", "CREEPER"), crit=["payload", "energy", "adapt"], prio=[.40, .30, .30], req={"payload": 8.0}),
    "lifter":  dict(types=("UGV",),           crit=["payload", "safety", "energy"], prio=[.50, .25, .25], req={"payload": 38.0}),
}
for _r, _d in ROLES.items():
    _d["M"] = pairwise_from_priorities(np.array(_d["prio"]))
    _d["W"], _d["CR"] = ahp_weights(_d["M"])
    assert _d["CR"] < 0.1, f"AHP profile {_r} inconsistent (CR={_d['CR']:.3f})"
    _d["idx"] = [CRIT.index(c) for c in _d["crit"]]
    _d["ben"] = BEN[_d["idx"]]
    _d["lo"] = np.array([min(TYPE_SPEC[t][i][0] for t in _d["types"]) for i in _d["idx"]])
    _d["hi"] = np.array([max(TYPE_SPEC[t][i][1] for t in _d["types"]) for i in _d["idx"]])
    _d["req_idx"] = [(CRIT.index(k), v) for k, v in _d["req"].items()]

# mission template: one 'scan-and-rescue' team, one supply task, one lifting task  (6 slots, 5 of them ground roles)
TEMPLATE = [("scout", 1), ("rescuer", 2)], [("carrier", 2)], [("lifter", 1)]
SLOTS_PER_TEMPLATE, GROUND_PER_TEMPLATE = 6, 5

def build_mission(k, priorities=(3, 2, 1)):
    """k replicated templates -> list of tasks: dict(id, prio, roles=[role,...]) ; priorities cycle 3,2,1 over tasks."""
    tasks = []
    for rep in range(k):
        for j, roles in enumerate(TEMPLATE):
            flat = [r for r, m in roles for _ in range(m)]
            tasks.append(dict(id=len(tasks), prio=priorities[j % len(priorities)], roles=flat, urgency=priorities[j % len(priorities)],
                              deadline=100 + 10 * j + rep))
    return tasks

def role_utility(caps, health, role):
    """Visible/true utility of robots for a role: weighted, design-range-normalised criteria, scaled by health. -> (N,)"""
    R = ROLES[role]
    x = np.clip((caps[:, R["idx"]] - R["lo"]) / (R["hi"] - R["lo"]), 0, 1)
    x = np.where(R["ben"], x, 1 - x)
    return (x @ R["W"]) * health

def eligibility(caps, types, available, role):
    R = ROLES[role]
    ok = np.isin(types, R["types"]) & available
    for ci, thr in R["req_idx"]:
        ok &= caps[:, ci] >= thr
    return ok

@dataclass
class Instance:
    N: int; seed: int
    types: np.ndarray; caps: np.ndarray; health: np.ndarray; mode: np.ndarray
    available: np.ndarray      # explicit availability / fault flags (hard constraint, known to every method)
    stale: np.ndarray          # telemetry older than the freshness bound  -> evidence status UNKNOWN
    dead: np.ndarray           # ground truth: stale AND actually down
    restricted: np.ndarray     # ground truth + knowledge: valid restriction record contradicts its operating mode in the zone
    tasks: list
    ids: np.ndarray = None      # persistent robot identifiers (used for deterministic tie-breaking)
    U: dict = field(default_factory=dict)   # role -> visible utility (N,)
    E: dict = field(default_factory=dict)   # role -> eligibility mask (N,)
    slots: list = field(default_factory=list)   # (task_id, role, prio)

def make_instance(N, seed, supply_ratio=None, templates=None, p_stale=0.06, p_restricted=0.05, p_unavail=0.04, type_frac=None):
    """Random instance. Demand structure is fixed (`templates`) unless a ground supply/demand ratio is requested."""
    rng = np.random.default_rng(MASTER_SEED + seed * 7 + N)
    n_t = {t: int(round(N * f)) for t, f in (type_frac or TYPE_FRAC).items()}; n_t["UGV"] += N - sum(n_t.values())
    types = np.array([t for t in TYPES for _ in range(n_t[t])])
    # robots come from a small model catalogue -> exact ties between units of the same model (realistic, and the source of CO multiplicity)
    caps = np.zeros((N, 6))
    for t in TYPES:
        m = np.where(types == t)[0]
        q = np.array([[0.12, 0.38, 0.63, 0.88]] * 6).T            # stratified quantiles, independently permuted per criterion
        for c in range(6): q[:, c] = rng.permutation(q[:, c])
        q = np.clip(q + rng.uniform(-0.05, 0.05, size=q.shape), 0, 1)
        catalogue = np.array([[lo + q[k, c] * (hi - lo) for c, (lo, hi) in enumerate(TYPE_SPEC[t])] for k in range(4)])
        pick = catalogue[rng.integers(0, 4, size=len(m))]
        jitter = np.where(rng.random((len(m), 1)) < 0.5, 0.0, rng.uniform(-0.05, 0.05, size=(len(m), 6)))
        caps[m] = pick * (1 + jitter)
    health = np.where(rng.random(N) < 0.85, 1.0, rng.uniform(0.6, 1.0, size=N))
    mode = (rng.random(N) < 0.25).astype(int)
    available = rng.random(N) >= p_unavail
    stale = rng.random(N) < p_stale
    dead = stale & (rng.random(N) < 0.5)
    restricted = rng.random(N) < p_restricted
    if templates is None:
        if supply_ratio is None: templates = 3
        else:
            n_ground = int((types != "UAV").sum())
            templates = max(1, int(round(n_ground / (GROUND_PER_TEMPLATE * supply_ratio))))
    tasks = build_mission(templates)
    inst = Instance(N, seed, types, caps, health, mode, available, stale, dead, restricted, tasks, ids=np.arange(N))
    for role in ROLES:
        inst.U[role] = role_utility(caps, health, role)
        inst.E[role] = eligibility(caps, types, available, role)
    inst.slots = [(t["id"], r, t["prio"]) for t in tasks for r in t["roles"]]
    return inst

def truly_feasible(inst, i):
    return (not inst.dead[i]) and (not inst.restricted[i])

def evaluate_assignment(inst, assign):
    """assign: list (len = #slots) of robot index or -1.  Returns the outcome metrics of the table."""
    n = len(inst.slots); a = np.asarray(assign)
    filled = a >= 0
    prio = np.array([s[2] for s in inst.slots], float); roles = [s[1] for s in inst.slots]
    util = np.array([inst.U[roles[j]][a[j]] if filled[j] else 0.0 for j in range(n)])
    ok = np.array([filled[j] and truly_feasible(inst, a[j]) for j in range(n)])
    # duplicates are invalid
    used = a[filled]; dup = len(used) - len(set(used.tolist()))
    # complete tasks
    task_ids = np.array([s[0] for s in inst.slots])
    complete = {t["id"]: bool(filled[task_ids == t["id"]].all()) for t in inst.tasks}
    complete_eff = {t["id"]: bool(ok[task_ids == t["id"]].all()) for t in inst.tasks}
    tp = {t["id"]: t["prio"] for t in inst.tasks}
    return dict(
        value=float((prio * util).sum()), value_eff=float((prio * util * ok).sum()),
        fill_rate=float(filled.mean()), fill_rate_eff=float(ok.mean()),
        task_fill_rate=float(np.mean(list(complete.values()))), task_fill_rate_eff=float(np.mean(list(complete_eff.values()))),
        unmet_pw=float((prio * (~filled)).sum()), unmet_pw_eff=float((prio * (~ok)).sum()),
        unmet_pw_norm=float((prio * (~filled)).sum() / prio.sum()), unmet_pw_eff_norm=float((prio * (~ok)).sum() / prio.sum()),
        invalid_rate=float((filled & ~ok).sum() / max(1, filled.sum())), duplicates=int(dup), n_slots=n,
        slots_filled=int(filled.sum()), assigned_set=tuple(sorted(inst.ids[used].tolist())),
    )

print({r: f"CR={d['CR']:.3f}" for r, d in ROLES.items()})


def permute_instance(inst, perm):
    """Same instance with the robot ARRAY ORDER shuffled (ids travel with the robots): tests order-independence."""
    p = np.asarray(perm)
    new = Instance(inst.N, inst.seed, inst.types[p], inst.caps[p], inst.health[p], inst.mode[p], inst.available[p], inst.stale[p],
                   inst.dead[p], inst.restricted[p], inst.tasks, ids=inst.ids[p])
    new.U = {r: u[p] for r, u in inst.U.items()}; new.E = {r: e[p] for r, e in inst.E.items()}; new.slots = inst.slots
    return new

## B. AhToVIK decision logic and versioned knowledge base
Task prioritisation, geometric-reference mapping (Eq. 1-2), compromise index (Eq. 3), bounded depth-first team formation, evidence gate.

In [ ]:
# ============================================================================
# B. AHTOVIK DECISION LOGIC (paper Sec. III-A..C)  +  VERSIONED KNOWLEDGE BASE (Sec. III-E)
#    Stage 1 task prioritisation (ready tasks by urgency, earliest deadline, id; context-aware criteria; AHP + CR check)
#    Stage 2 geometric-reference mapping (Eq. 1-2: vector normalisation, weighting, PIS/NIS, D+/D-)
#    Stage 3 compromise ranking (Eq. 3) + bounded depth-first feasible team formation + evidence gate
# ============================================================================
SUPPORTED, CONTRADICTED, UNKNOWN = "supported", "contradicted", "unknown"

class KnowledgeBase:
    """Versioned runtime knowledge: evidence assertions (status + validity interval), decisions, verified recovery cases."""
    def __init__(self, retention=20, k_min=2, reuse=True):
        self.version = 0; self.assertions = {}; self.decisions = []; self.cases = []
        self.retention, self.k_min, self.reuse = retention, k_min, reuse
        self.n_reads = 0; self.n_writes = 0; self.episode = 0

    def new_episode(self, ep):
        """New mission: transient evidence expires, verified recovery cases persist (retention-limited)."""
        self.assertions = {}; self.decisions = []; self.episode = ep; self.version += 1

    # --- versioned snapshot: every decision references the version it read
    def snapshot(self):
        self.n_reads += 1; return self.version

    def revalidate(self, snap):
        """True if no relevant record changed since the snapshot (before commit the planner revalidates)."""
        self.n_reads += 1; return snap == self.version

    # --- evidence assertions: predicate/scope/status/validity interval
    def assert_(self, pred, scope, status, now, ttl=60):
        self.assertions[(pred, scope)] = (status, now + ttl); self.version += 1; self.n_writes += 1

    def status(self, pred, scope, now):
        v = self.assertions.get((pred, scope))
        if v is None or now >= v[1]: return UNKNOWN          # expired evidence cannot authorise actions
        return v[0]

    def record_decision(self, d):
        self.decisions.append(d); self.version += 1; self.n_writes += 1

    # --- recovery cases (verified outcomes only) and diagnostic selection (dynamic benchmark / RQ3)
    def add_case(self, ctx, cause, diags, action, duration, episode):
        self.cases.append(dict(ctx=ctx, cause=cause, diags=tuple(diags), action=action, duration=duration, ep=episode))
        self.version += 1; self.n_writes += 1

    def matching_cases(self, ctx, episode):
        self.n_reads += 1
        return [c for c in self.cases if c["ctx"] == ctx and episode - c["ep"] <= self.retention]

    def choose_diag(self, ctx, diag_specs, episode):
        """diag_specs: {name: duration}. Default order = (declared duration, name). With >= k_min validated matching cases the
        planner orders diagnostics by their expected time to resolve the uncertainty given the observed cause frequencies
        (historical knowledge changes the diagnostic choice only; it never authorises a replacement)."""
        default = sorted(diag_specs, key=lambda d: (diag_specs[d], d))
        if not self.reuse: return default
        cs = self.matching_cases(ctx, episode)
        if len(cs) < self.k_min: return default
        p_restr = np.mean([c["cause"] == "restriction" for c in cs])
        # time to resolution when diagnostic X is tried first: mode_check always resolves; self_test resolves only isolated faults
        t_mode = diag_specs["mode_check"]; t_self = diag_specs["self_test"]
        e_self_first = t_self + p_restr * t_mode
        e_mode_first = t_mode
        return ["mode_check", "self_test"] if e_mode_first < e_self_first else ["self_test", "mode_check"]

    def size_bytes(self):
        return len(json.dumps([self.cases, self.decisions[-200:]], default=str))

# ---------------------------------------------------------------------------- Stage 2: geometric-reference mapping
def _mm(x):
    r = x.max() - x.min()
    return np.zeros_like(x) if r < 1e-12 else (x - x.min()) / r          # zero spread contributes zero

def geometric_distances(X, w, ben):
    """Eq. (1)-(2): vector normalisation, weighting, positive/negative ideal, Euclidean distances. X: (m, n)."""
    nrm = np.sqrt((X ** 2).sum(0)); nrm = np.where(nrm == 0, 1.0, nrm)  # zero-norm column carries no information
    Y = (X / nrm) * w
    pis = np.where(ben, Y.max(0), Y.min(0)); nis = np.where(ben, Y.min(0), Y.max(0))
    return np.sqrt(((Y - pis) ** 2).sum(1)), np.sqrt(((Y - nis) ** 2).sum(1))

def compromise_index(dp, dm, v=0.5):
    """Eq. (3): modified VIKOR index, lower = better."""
    return v * _mm(dp) + (1 - v) * (1 - _mm(dm))

# ---------------------------------------------------------------------------- Stage 3: bounded DFS team formation
def dfs_team(role_order, counts, ranked, node_limit, compat=None):
    """role_order: roles (most constrained first); counts: role -> multiplicity; ranked: role -> robot ids in ascending Q.
    Returns (team: role -> [robots] | None, nodes, incomplete_flag). Symmetric permutations of a repeated role are skipped."""
    slots = [r for r in role_order for _ in range(counts[r])]
    chosen = []; used = set(); nodes = [0]; hit_limit = [False]
    def rec(k, start):
        if k == len(slots): return True
        role = slots[k]; order = ranked[role]
        for pos in range(start.get(role, 0), len(order)):
            r = order[pos]
            if r in used: continue
            nodes[0] += 1
            if nodes[0] > node_limit: hit_limit[0] = True; return False
            if compat is not None and not all(compat(r, c) for c, _ in chosen): continue
            chosen.append((r, role)); used.add(r)
            nxt = dict(start); nxt[role] = pos + 1
            if rec(k + 1, nxt): return True
            chosen.pop(); used.discard(r)
            if hit_limit[0]: return False
        return False
    ok = rec(0, {})
    if not ok: return None, nodes[0], hit_limit[0]
    team = defaultdict(list)
    for r, role in chosen: team[role].append(int(r))
    return dict(team), nodes[0], False

@dataclass(frozen=True)
class AhtovikCfg:
    v: float = 0.5               # compromise weight in Eq. (3)
    node_limit: int = 20000      # bounded-search budget of the depth-first team formation
    use_gate: bool = True        # evidence gate before dispatch (stale -> unknown, contradicted -> inadmissible)
    allow_partial: bool = False  # faithful to the paper: a task without a complete feasible team stays pending

def ahtovik_assign(inst, cfg=AhtovikCfg(), kb=None, compat=None, v=None):
    """Full static AHTOVIK decision: returns dict(assign, build_ms, solve_ms, nodes, incomplete, pending)."""
    v = cfg.v if v is None else v
    kb = kb if kb is not None else KnowledgeBase()
    N = inst.N; assigned = np.zeros(N, bool); out = {}
    slot_of = defaultdict(list)
    for j, (tid, role, pr) in enumerate(inst.slots): slot_of[tid].append(j)
    t_build = t_solve = 0.0; nodes_total = 0; incomplete = 0; pending = []
    tb = time.perf_counter()
    # Stage 1: context-aware criteria selection + AHP weights + consistency circuit breaker (re-derived per decision)
    W = {}
    for role, R in ROLES.items():
        w, cr = ahp_weights(R["M"])
        if cr >= 0.1: raise RuntimeError(f"AHP profile {role} failed the consistency check")
        W[role] = w
    tasks = sorted(inst.tasks, key=lambda t: (-t["urgency"], t["deadline"], t["id"]))     # ready tasks: urgency, deadline, id
    t_build += time.perf_counter() - tb
    for task in tasks:
        snap = kb.snapshot()
        tb = time.perf_counter()
        counts = defaultdict(int)
        for r in task["roles"]: counts[r] += 1
        ranked = {}
        for role in counts:
            R = ROLES[role]
            pool = inst.E[role] & ~assigned
            if cfg.use_gate:                                   # prerequisites must be SUPPORTED: stale -> unknown, restricted -> contradicted
                pool = pool & ~inst.stale & ~inst.restricted
            idx = np.where(pool)[0]
            if len(idx) == 0: ranked[role] = idx; continue
            dp, dm = geometric_distances(inst.caps[np.ix_(idx, R["idx"])], W[role], R["ben"])
            Q = compromise_index(dp, dm, v)
            ranked[role] = idx[np.lexsort((inst.ids[idx], Q))]   # ascending Q, ties by robot identifier
        t_build += time.perf_counter() - tb
        ts = time.perf_counter()
        feasible = all(len(ranked[r]) >= counts[r] for r in counts)
        team = None
        if feasible:
            order = sorted(counts, key=lambda r: len(ranked[r]) / counts[r])                   # most constrained role first
            team, nodes, hit = dfs_team(order, counts, ranked, cfg.node_limit, compat)
            nodes_total += nodes; incomplete += int(hit)
        if team is not None and cfg.use_gate:
            # evidence gate: revalidate every prerequisite of the proposal against current Knowledge before dispatch
            ok = all(inst.available[r] and not inst.stale[r] and not inst.restricted[r] for rs in team.values() for r in rs)
            if not ok: team = None
        if team is None:
            pending.append(task["id"])
        else:
            used = {role: 0 for role in team}
            for j in slot_of[task["id"]]:
                role = inst.slots[j][1]; r = team[role][used[role]]; used[role] += 1
                out[j] = r; assigned[r] = True
            kb.record_decision(dict(task=task["id"], team=team, snapshot=snap))
        t_solve += time.perf_counter() - ts
    if cfg.allow_partial and pending:                       # optional extension: best-effort staffing of pending tasks
        ts = time.perf_counter()
        for tid in pending:
            for j in slot_of[tid]:
                role = inst.slots[j][1]; R = ROLES[role]
                pool = inst.E[role] & ~assigned
                if cfg.use_gate: pool = pool & ~inst.stale & ~inst.restricted
                idx = np.where(pool)[0]
                if len(idx) == 0: continue
                dp, dm = geometric_distances(inst.caps[np.ix_(idx, R["idx"])], W[role], R["ben"])
                r = idx[np.lexsort((inst.ids[idx], compromise_index(dp, dm, v)))[0]]
                out[j] = int(r); assigned[r] = True
        t_solve += time.perf_counter() - ts
    assign = [out.get(j, -1) for j in range(len(inst.slots))]
    return dict(assign=assign, build_ms=1e3 * t_build, solve_ms=1e3 * t_solve, nodes=nodes_total, incomplete=incomplete, pending=pending)

## C. Static baselines and the CO reference solver (MILP with BIG_M)

In [ ]:
# ============================================================================
# C. STATIC BASELINES / REFERENCE SOLVERS
#    CO         : exact combinatorial optimisation (MILP, HiGHS) with a BIG_M fill-first term  -> certified ceiling
#                 (single-task / single-robot / instantaneous-assignment problem of Gerkey & Mataric [13])
#    SAW/TOPSIS/VIKOR-greedy : the MCDM rankers of [15][16] plugged into the same priority-ordered slot filling
#    TRAIT      : capability-coverage matching in the spirit of trait-based resilient allocation [5] / Multi-3 [14]
#    SSI        : sequential single-item auction on the visible utility (market-based reference)
#    RANDOM     : random eligible robot (lower bound)
# ============================================================================
def _priority_slots(inst):
    """Slots in the same deterministic priority order AHTOVIK uses: urgency, deadline, task id."""
    order = sorted(inst.tasks, key=lambda t: (-t["urgency"], t["deadline"], t["id"]))
    by_task = defaultdict(list)
    for j, (tid, role, pr) in enumerate(inst.slots): by_task[tid].append(j)
    return [j for t in order for j in by_task[t["id"]]]

# ---------------------------------------------------------------- CO: MILP with BIG_M
def co_assign(inst, big_m=1e6, nudge=0.0, rng=None, time_limit=120.0, pure_value=False):
    t0 = time.perf_counter()
    n = len(inst.slots); N = inst.N
    rows, cols, val, pr = [], [], [], []
    elig_idx = {r: np.where(inst.E[r])[0] for r in ROLES}
    for j, (tid, role, p) in enumerate(inst.slots):
        e = elig_idx[role]
        rows.append(np.full(len(e), j)); cols.append(e); val.append(p * inst.U[role][e]); pr.append(np.full(len(e), float(p)))
    if not any(len(c) for c in cols):
        return dict(assign=[-1] * n, build_ms=0.0, solve_ms=0.0, status="empty", obj=0.0)
    rows = np.concatenate(rows); cols = np.concatenate(cols); val = np.concatenate(val); pr = np.concatenate(pr)
    if nudge > 0: val = val + (rng or np.random.default_rng()).normal(0, nudge, size=val.shape)   # negligible objective perturbation
    coef = val + (0.0 if pure_value else big_m * pr)
    nv = len(rows)
    uniq, inv = np.unique(cols, return_inverse=True)
    r_idx = np.concatenate([rows, n + inv]); c_idx = np.concatenate([np.arange(nv), np.arange(nv)])
    A = sparse.csr_matrix((np.ones(2 * nv), (r_idx, c_idx)), shape=(n + len(uniq), nv))
    build = time.perf_counter() - t0
    t1 = time.perf_counter()
    res = milp(c=-coef, constraints=LinearConstraint(A, -np.inf, 1.0), integrality=np.ones(nv), bounds=Bounds(0, 1),
               options=dict(time_limit=time_limit, presolve=True))
    solve = time.perf_counter() - t1
    assign = [-1] * n
    if res.x is not None:
        for k in np.where(res.x > 0.5)[0]: assign[rows[k]] = int(cols[k])
    return dict(assign=assign, build_ms=1e3 * build, solve_ms=1e3 * solve, status=res.status, obj=(-res.fun if res.fun is not None else np.nan))

# ---------------------------------------------------------------- scorers for the greedy MCDM baselines (higher = better)
def sc_saw(X, w, ben):
    R = np.where(ben, X / np.where(X.max(0) == 0, 1, X.max(0)), X.min(0) / np.where(X == 0, 1e-9, X))
    return R @ w

def sc_topsis(X, w, ben):
    dp, dm = geometric_distances(X, w, ben)
    return dm / np.where(dp + dm == 0, 1, dp + dm)

def sc_vikor(X, w, ben, v=0.5):
    best = np.where(ben, X.max(0), X.min(0)); worst = np.where(ben, X.min(0), X.max(0))
    den = np.where(np.abs(best - worst) < 1e-12, 1.0, best - worst)
    d = w * (best - X) / den
    return -(v * _mm(d.sum(1)) + (1 - v) * _mm(d.max(1)))

def sc_trait(X, w, ben, lo=None, hi=None):
    """Trait-coverage: how much of the required (design-midpoint) trait level a robot covers, weighted by criterion importance."""
    ref = 0.5 * (lo + hi)
    cov = np.where(ben, np.minimum(X / ref, 1.0), np.minimum(ref / np.where(X == 0, 1e-9, X), 1.0))
    return cov @ w

def greedy_assign(inst, scorer, v=None, use_role_range=False):
    t0 = time.perf_counter()
    N = inst.N; assigned = np.zeros(N, bool); assign = [-1] * len(inst.slots)
    build = solve = 0.0
    W = {r: ahp_weights(R["M"])[0] for r, R in ROLES.items()}
    build += time.perf_counter() - t0
    for j in _priority_slots(inst):
        role = inst.slots[j][1]; R = ROLES[role]
        tb = time.perf_counter()
        idx = np.where(inst.E[role] & ~assigned)[0]
        if len(idx) == 0: build += time.perf_counter() - tb; continue
        X = inst.caps[np.ix_(idx, R["idx"])]
        kw = {}
        if scorer is sc_trait: kw = dict(lo=R["lo"], hi=R["hi"])
        if v is not None and scorer is sc_vikor: kw["v"] = v
        sc = scorer(X, W[role], R["ben"], **kw)
        build += time.perf_counter() - tb
        ts = time.perf_counter()
        k = idx[np.lexsort((inst.ids[idx], -sc))[0]]
        assign[j] = int(k); assigned[k] = True
        solve += time.perf_counter() - ts
    return dict(assign=assign, build_ms=1e3 * build, solve_ms=1e3 * solve)

def ssi_assign(inst):
    t0 = time.perf_counter(); N = inst.N; assigned = np.zeros(N, bool); assign = [-1] * len(inst.slots)
    build = time.perf_counter() - t0; solve = 0.0
    for j in _priority_slots(inst):
        role = inst.slots[j][1]; ts = time.perf_counter()
        idx = np.where(inst.E[role] & ~assigned)[0]
        if len(idx):
            k = idx[np.lexsort((inst.ids[idx], -inst.U[role][idx]))[0]]       # highest bid (visible utility) wins this item
            assign[j] = int(k); assigned[k] = True
        solve += time.perf_counter() - ts
    return dict(assign=assign, build_ms=1e3 * build, solve_ms=1e3 * solve)

def random_assign(inst, seed=0):
    rng = np.random.default_rng(seed); assigned = np.zeros(inst.N, bool); assign = [-1] * len(inst.slots)
    for j in _priority_slots(inst):
        idx = np.where(inst.E[inst.slots[j][1]] & ~assigned)[0]
        if len(idx): k = int(rng.choice(idx)); assign[j] = k; assigned[k] = True
    return dict(assign=assign, build_ms=0.0, solve_ms=0.0)

# ---------------------------------------------------------------- method registry
STATIC_METHODS = {
    "AHTOVIK":        lambda inst, **kw: ahtovik_assign(inst, AhtovikCfg(), **kw),
    "AHTOVIK-core":   lambda inst, **kw: ahtovik_assign(inst, AhtovikCfg(use_gate=False), **kw),        # ranking + team formation, no evidence gate
    "AHTOVIK-partial": lambda inst, **kw: ahtovik_assign(inst, AhtovikCfg(allow_partial=True), **kw),    # best-effort staffing of pending tasks
    "CO-MILP":        lambda inst, **kw: co_assign(inst, **kw),
    "SAW-greedy":     lambda inst, **kw: greedy_assign(inst, sc_saw),
    "TOPSIS-greedy":  lambda inst, **kw: greedy_assign(inst, sc_topsis),
    "VIKOR-greedy":   lambda inst, **kw: greedy_assign(inst, sc_vikor, **kw),
    "TRAIT-greedy":   lambda inst, **kw: greedy_assign(inst, sc_trait),
    "SSI-auction":    lambda inst, **kw: ssi_assign(inst),
    "RANDOM":         lambda inst, **kw: random_assign(inst, **kw),
}
def run_static(name, inst, **kw):
    t0 = time.perf_counter(); r = STATIC_METHODS[name](inst, **kw); total = 1e3 * (time.perf_counter() - t0)
    m = evaluate_assignment(inst, r["assign"])
    m.update(method=name, build_ms=r.get("build_ms", np.nan), solve_ms=r.get("solve_ms", np.nan), total_ms=total)
    for k in ("nodes", "incomplete", "status", "obj"):
        if k in r: m[k] = r[k]
    return m

## D. Static benchmark runners

In [ ]:
# ============================================================================
# D. STATIC DECISION BENCHMARK  - the evaluation factors of the table, one case each
#    Latency | Scalability | Optimality | Feasibility/fill-rate | Priority-preservation under scarcity |
#    Robustness/sensitivity (v, BIG_M) | Determinism/solution multiplicity | Implementation complexity (later cell)
# ============================================================================
CORE_METHODS = ["AHTOVIK", "AHTOVIK-core", "AHTOVIK-partial", "CO-MILP", "SAW-greedy", "TOPSIS-greedy", "VIKOR-greedy",
                "TRAIT-greedy", "SSI-auction", "RANDOM"]
SCALE_METHODS = ["AHTOVIK", "AHTOVIK-core", "CO-MILP", "SAW-greedy", "TOPSIS-greedy", "VIKOR-greedy", "TRAIT-greedy", "SSI-auction"]
SCARCITY = [0.8, 1.0, 1.5, 3.0, 10.0]
VS = [0.0, 0.25, 0.5, 0.75, 1.0]
BIGMS = [1e3, 1e4, 1e6, 1e9]
N_CORE, N_SWEEP = 100, 200

def _static_core_one(seed):
    inst = make_instance(N_CORE, seed); rows = []
    ref_pure = evaluate_assignment(inst, co_assign(inst, pure_value=True)["assign"])["value"]
    for m in CORE_METHODS:
        r = run_static(m, inst, **({"seed": seed} if m == "RANDOM" else {})); r.update(seed=seed, co_pure_value=ref_pure); rows.append(r)
    return rows

def _scarcity_one(args):
    ratio, seed = args
    inst = make_instance(N_SWEEP, seed, supply_ratio=ratio); rows = []
    for m in CORE_METHODS:
        if m == "RANDOM": continue
        r = run_static(m, inst); r.update(seed=seed, ratio=ratio, n_slots=len(inst.slots)); rows.append(r)
    return rows

def _scale_one(args):
    N, seed = args
    inst = make_instance(N, seed); rows = []
    for m in SCALE_METHODS:
        r = run_static(m, inst); r.update(seed=seed, N=N); rows.append(r)
    return rows

def _jaccard(a, b):
    a, b = set(a), set(b)
    return 1.0 if not (a | b) else len(a & b) / len(a | b)

def _robust_v_one(seed):
    inst = make_instance(N_CORE, seed); rows = []
    role = "carrier"; R = ROLES[role]
    pool = np.where(inst.E[role] & ~inst.stale & ~inst.restricted)[0]
    X = inst.caps[np.ix_(pool, R["idx"])]; dp, dm = geometric_distances(X, R["W"], R["ben"])
    ref_sets = {}
    for method in ["AHTOVIK", "VIKOR-greedy"]:
        ref_rank = (-compromise_index(dp, dm, 0.5)) if method == "AHTOVIK" else sc_vikor(X, R["W"], R["ben"], 0.5)
        for v in VS:
            sc = (-compromise_index(dp, dm, v)) if method == "AHTOVIK" else sc_vikor(X, R["W"], R["ben"], v)
            tau = kendall(sc, ref_rank); top1 = float(np.argmax(sc) == np.argmax(ref_rank))
            a = (ahtovik_assign(inst, AhtovikCfg(), v=v)["assign"] if method == "AHTOVIK" else greedy_assign(inst, sc_vikor, v=v)["assign"])
            m = evaluate_assignment(inst, a); ref_sets.setdefault(method, {})[v] = m["assigned_set"]
            rows.append(dict(seed=seed, method=method, v=v, tau=tau, top1_same=top1, value=m["value"], assigned_set=m["assigned_set"]))
    for r in rows: r["jaccard_vs_v05"] = _jaccard(r["assigned_set"], ref_sets[r["method"]][0.5]); r.pop("assigned_set")
    return rows

def _robust_bigm_one(seed):
    inst = make_instance(N_CORE, seed); rows = []; sets = {}
    pure = evaluate_assignment(inst, co_assign(inst, pure_value=True)["assign"])["value"]
    for bm in BIGMS:
        r = co_assign(inst, big_m=bm); m = evaluate_assignment(inst, r["assign"]); sets[bm] = m["assigned_set"]
        rows.append(dict(seed=seed, big_m=bm, value=m["value"], fill=m["fill_rate"], status=r["status"], pure_value=pure, set=m["assigned_set"]))
    for r in rows: r["jaccard_vs_1e6"] = _jaccard(r["set"], sets[1e6]); r["changed_vs_1e6"] = float(r["set"] != sets[1e6]); r.pop("set")
    return rows

def _determinism_one(seed, n_nudge=20, n_perm=3):
    inst = make_instance(N_CORE, seed); rng = np.random.default_rng(seed + 99)
    ref = evaluate_assignment(inst, co_assign(inst)["assign"])
    ch = []
    for k in range(n_nudge):
        a = co_assign(inst, nudge=1e-9, rng=rng)["assign"]; ch.append(evaluate_assignment(inst, a)["assigned_set"] != ref["assigned_set"])
    rows = [dict(seed=seed, test="CO objective nudge (1e-9)", method="CO-MILP", changed_any=float(any(ch)), changed_frac=float(np.mean(ch)))]
    for m in CORE_METHODS:
        if m == "RANDOM": continue
        base = run_static(m, inst)["assigned_set"]; ch2 = []
        for k in range(n_perm):
            perm = rng.permutation(inst.N); ip = permute_instance(inst, perm)
            ch2.append(run_static(m, ip)["assigned_set"] != base)
        rows.append(dict(seed=seed, test="robot-order permutation", method=m, changed_any=float(any(ch2)), changed_frac=float(np.mean(ch2))))
    return rows

def savefig(name):
    plt.tight_layout(); plt.savefig(f"{OUT}/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

def kendall(a, b):
    if len(a) < 2: return 1.0
    t = stats.kendalltau(a, b)[0]
    return 1.0 if np.isnan(t) else t

def flat(list_of_lists): return [x for l in list_of_lists for x in l]

def run_static_cases():
    out = {}
    out["core"] = cached("static_core", lambda: pd.DataFrame(flat(pmap(_static_core_one, list(range(N_RUNS))))))
    out["scarcity"] = cached("static_scarcity", lambda: pd.DataFrame(flat(pmap(_scarcity_one, [(r, s) for r in SCARCITY for s in range(N_RUNS)]))))
    def scale():
        rows = []
        for N in SCALE_N:                      # sequential over N so that timings are not distorted by parallel load
            rows += flat([_scale_one((N, s)) for s in range(SCALE_RUNS[N])])
        return pd.DataFrame(rows)
    out["scale"] = cached("static_scale", scale)
    out["robust_v"] = cached("static_robust_v", lambda: pd.DataFrame(flat(pmap(_robust_v_one, list(range(N_RUNS))))))
    out["robust_bigm"] = cached("static_robust_bigm", lambda: pd.DataFrame(flat(pmap(_robust_bigm_one, list(range(N_RUNS))))))
    out["determinism"] = cached("static_determinism", lambda: pd.DataFrame(flat(pmap(_determinism_one, list(range(N_RUNS))))))
    return out

# ----------------------------------------------------------------------------- analysis helpers
def ci(x, alpha=0.05, n=1500, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 3: return (np.nan, np.nan)
    rng = np.random.default_rng(seed); m = rng.choice(x, size=(n, len(x))).mean(1)
    return tuple(np.quantile(m, [alpha / 2, 1 - alpha / 2]))

def mean_ci(df, by, cols):
    rows = []
    for key, g in df.groupby(by):
        key = key if isinstance(key, tuple) else (key,)
        row = dict(zip(by, key)); row["n"] = len(g)
        for c in cols:
            lo, hi = ci(g[c]); row[c] = g[c].mean(); row[c + "_lo"] = lo; row[c + "_hi"] = hi
        rows.append(row)
    return pd.DataFrame(rows)

def loglog_slope(Ns, ts, lo=None, hi=None):
    Ns = np.asarray(Ns, float); ts = np.asarray(ts, float)
    m = np.ones(len(Ns), bool)
    if lo is not None: m &= Ns >= lo
    if hi is not None: m &= Ns <= hi
    if m.sum() < 2: return np.nan
    return stats.linregress(np.log10(Ns[m]), np.log10(ts[m])).slope

## E. Dynamic world (plant, telemetry, leases, checkpoints, diagnostics, metrics)

In [ ]:
# ============================================================================
# E. DYNAMIC WORLD (plant + monitoring substrate shared by AHTOVIK and all baselines)
#    Truth is hidden from controllers; they only see telemetry (noisy, delayed by loss), explicit fault flags and
#    accepted-progress observations.  Differences between methods therefore come from the DECISION LOGIC only.
#    Features that the paper relies on: checkpoints (+ compatibility), expiring execution authority (leases), atomic
#    reservation, zone/mode restrictions with ambiguous symptoms, permitted diagnostics, task dependencies, deadlines.
# ============================================================================
INF = 10 ** 9
H_MIN = 0.5
DIAG_SPECS = {"self_test": 3, "mode_check": 6}      # declared maximum durations of the permitted diagnostics (steps)

@dataclass(frozen=True)
class DynCfg:
    T: int = 170
    sigma: float = 0.05           # relative telemetry noise (Gaussian)
    work: float = 60.0            # work units per slot
    handover: int = 3             # steps of no progress while a replacement takes over
    deadline: int = 115           # task deadline dT
    lease: int = 8                # execution-authority lease length (steps) for methods that use leases
    clock_margin: int = 2         # clock-uncertainty margin before a lapsed authority is considered expired
    conflict_penalty: int = 5     # slot downtime when a replacement collides with a still-active old executor
    verify_delay: int = 3         # steps until a duplicate dispatch is detected
    p_incompat: float = 0.25      # share of robots that cannot resume another robot's checkpoint (restart needed)
    restart_loss: float = 0.5     # fraction of progress lost on a restart
    diag_acc: float = 0.95        # probability that a diagnostic returns a conclusive result
    loss_prob: float = 0.0        # background telemetry packet loss
    p_mode1: float = 0.10         # share of robots using the alternative operating mode
    p_switch: float = 0.20        # share of robots with an approved alternative mode (reconfigure-incumbent option)
    n_robots: int = 28

class Controller:
    """Base class of every decision method.  atomic: reservation through a coordination service; lease: expiring authority."""
    name = "ctl"; atomic = False; lease = False; uses_kb = False
    def init(self, w): raise NotImplementedError
    def step(self, w, t): raise NotImplementedError

def role_weights_dyn(role):
    """AHP weights of the role's criteria, plus the observed health criterion (weight 0.2)."""
    return np.concatenate([ROLES[role]["W"] * 0.8, [0.2]])

class World:
    def __init__(self, inst, events, cfg, ctrl, seed, kb=None, attrs=None):
        self.inst, self.cfg, self.ctrl, self.kb = inst, cfg, ctrl, kb
        n = inst.N; self.n = n; self.types = inst.types; self.caps = inst.caps; self.ids = inst.ids
        self.rng = np.random.default_rng(seed); self.rng2 = np.random.default_rng(seed + 7919)
        r0 = np.random.default_rng(inst.seed * 31 + 7)                  # static robot attributes: identical for every method
        self.mode = (r0.random(n) < cfg.p_mode1).astype(int); self.h0 = inst.health.copy()
        self.ckpt_ok = r0.random(n) >= cfg.p_incompat; self.can_switch = r0.random(n) < cfg.p_switch
        if attrs:                                                         # scenario-specific robot attributes (e.g. knowledge-loop streams)
            if 'mode' in attrs: self.mode = np.asarray(attrs['mode']).copy()
            if 'can_switch' in attrs: self.can_switch = np.asarray(attrs['can_switch']).copy()
        self.E = inst.E; self.u0 = {r: role_utility(self.caps, np.ones(n), r) for r in ROLES}
        self.tasks = inst.tasks; self.zone_of = {t["id"]: t["id"] % 3 for t in inst.tasks}
        self.slots = []
        for t in inst.tasks:
            for k, role in enumerate(t["roles"]):
                self.slots.append(dict(tid=t["id"], k=k, role=role, prio=t["prio"], zone=self.zone_of[t["id"]], deadline=cfg.deadline,
                                       exec=-1, ho=0, prog=0.0, done=None, rho=1.0, rho_hist=deque(maxlen=8), lock=-1, sw=0, diag=None,
                                       esc_until=-1, lease_last=0, idle=0, idle_fault=False, reass=0, inf_since=None, st={}))
        self.slots.sort(key=lambda s: (-s["prio"], s["tid"], s["k"]))
        self.slot_by = {(s["tid"], s["k"]): s for s in self.slots}
        self.ev_by_t = defaultdict(list)
        for e in events: self.ev_by_t[e["t"]].append(dict(e))
        self.failed = np.zeros(n, bool); self.snag = np.zeros(n, bool); self.fault_flag_t = np.full(n, INF)
        self.dips, self.ramps, self.drops, self.restr = [], [], [], []
        self.last_caps = self.caps.copy(); self.last_h = self.h0.copy(); self.last_t = np.zeros(n, int)
        self.tele = np.ones(n, bool); self.h = self.h0.copy()
        self.queue = []; self.c = defaultdict(float); self.incidents = []; self.bad_causes = set()
        self.t = 0; self.t_decide = 0.0; self.n_decisions = 0; self.progress_deadline = None
        self.diag_results = []

    # ------------------------------------------------------------------ truth
    def _resolve(self, tgt):
        if isinstance(tgt, (int, np.integer)): return int(tgt)
        kind, tid, k = tgt.split(":"); s = self.slot_by[(int(tid), int(k))]
        if kind == "inc": return s["exec"] if (s["done"] is None and s["exec"] >= 0) else None
        if kind == "top_standby":
            taken = {x["exec"] for x in self.slots if x["exec"] >= 0}
            ok = [i for i in range(self.n) if self.E[s["role"]][i] and i not in taken and not self.failed[i]]
            return max(ok, key=lambda i: self.u0[s["role"]][i]) if ok else None
        raise ValueError(tgt)

    def _slot_of_exec(self, r):
        for s in self.slots:
            if s["exec"] == r and s["done"] is None: return s
        return None

    def _apply_events(self, t):
        for e in self.ev_by_t.get(t, []):
            r = self._resolve(e["target"]) if "target" in e else None
            if r is None: continue
            k = e["kind"]
            if k == "dip": self.dips.append((r, t, t + e["dur"], e["level"]))
            elif k == "dropout": self.drops.append((r, t, t + e["dur"]))
            elif k == "ramp": self.ramps.append((r, t, e["length"], e["final"]))
            elif k == "hard": self.failed[r] = True; self.fault_flag_t[r] = t + 1
            elif k == "silent": self.failed[r] = True
            elif k == "isolated": self.snag[r] = True                                   # robot-level stall (healthy self-diagnosis)
            elif k == "restrict":                                                       # zone/mode restriction hits every robot in that mode
                s = self._slot_of_exec(r)
                if s is not None: self.restr.append((s["zone"], int(self.mode[r]), t, t + e["dur"]))

    def _health(self, t):
        h = self.h0.copy()
        for r, t0, t1, lvl in self.dips:
            if t0 <= t < t1: h[r] = min(h[r], lvl)
        for r, t0, L, fin in self.ramps:
            if t >= t0: h[r] = min(h[r], 1 - (1 - fin) * min(1.0, (t - t0) / L))
        h[self.failed] = 0.0
        return h

    def restricted_now(self, zone, mode, t=None):
        t = self.t if t is None else t
        return any(z == zone and m == mode and t0 <= t < t1 for z, m, t0, t1 in self.restr)

    def infeasible_for(self, r, s, t=None):
        t = self.t if t is None else t
        return bool(self.failed[r] or self.snag[r] or self.h[r] < H_MIN or self.restricted_now(s["zone"], self.mode[r], t))

    def needs_replacement(self, r, s, t=None):
        return self.infeasible_for(r, s, t) or any(rr == r and t0 <= self.t for rr, t0, _, _ in self.ramps)

    def contained(self, s):
        """Lease-based methods: a robot that could not renew its authority stops by itself (containment)."""
        return self.ctrl.lease and (self.t - s["lease_last"]) >= self.cfg.lease

    def authority_expired(self, s):
        return (self.t - s["lease_last"]) >= self.cfg.lease + self.cfg.clock_margin

    # ------------------------------------------------------------------ monitoring
    def _telemetry(self, t):
        n, sg = self.n, self.cfg.sigma
        nc = self.rng.standard_normal((n, 6)); nh = self.rng.standard_normal(n); nr = self.rng.standard_normal(n); lu = self.rng.random(n)
        drop = np.zeros(n, bool)
        for r, t0, t1 in self.drops:
            if t0 <= t < t1: drop[r] = True
        self.tele = (~self.failed) & (~drop) & (lu >= self.cfg.loss_prob)
        m = self.tele
        self.last_caps[m] = self.caps[m] * (1 + sg * nc[m]); self.last_h[m] = np.clip(self.h[m] + sg * nh[m], 0, 1.2); self.last_t[m] = t
        for s in self.slots:
            r = s["exec"]
            if r < 0 or s["done"] is not None: continue
            if self.tele[r]: s["lease_last"] = t                        # authority renewed
            if s["ho"] == 0 and s["sw"] == 0 and self.tele[r]:
                eff = 0.0 if self.infeasible_for(r, s, t) else float(self.h[r])
                s["rho"] = eff + sg * nr[r]; s["rho_hist"].append(s["rho"])

    def age(self, r): return self.t - self.last_t[r]
    def flagged(self, r): return self.fault_flag_t[r] <= self.t
    def taken(self): return {s["exec"] for s in self.slots if s["exec"] >= 0}
    def pool(self, s, fresh=None, exclude=()):
        ok = self.E[s["role"]].copy()
        for i in self.taken(): ok[i] = False
        for i in exclude: ok[i] = False
        ok &= (self.fault_flag_t > self.t)
        if fresh is not None: ok &= (self.t - self.last_t) <= fresh
        return np.where(ok)[0]

    def feat(self, role, idx):
        """Observed decision matrix of candidates idx: role criteria + last-known health."""
        R = ROLES[role]
        return np.column_stack([self.last_caps[np.ix_(idx, R["idx"])], self.last_h[idx]])

    def ben(self, role): return np.concatenate([ROLES[role]["ben"], [True]])

    def q_rank(self, s, idx, v=0.5):
        """AHTOVIK compromise ranking of candidates idx for slot s -> ids ordered best-first."""
        if len(idx) == 0: return idx
        dp, dm = geometric_distances(self.feat(s["role"], idx), role_weights_dyn(s["role"]), self.ben(s["role"]))
        return idx[np.lexsort((self.ids[idx], compromise_index(dp, dm, v)))]

    def util_est(self, s, idx):
        """Visible utility estimate used by utility-based baselines (observed capabilities x observed health)."""
        R = ROLES[s["role"]]
        x = np.clip((self.last_caps[np.ix_(idx, R["idx"])] - R["lo"]) / (R["hi"] - R["lo"]), 0, 1)
        return (np.where(R["ben"], x, 1 - x) @ R["W"]) * np.clip(self.last_h[idx], 0, 1)

    # ------------------------------------------------------------------ actions available to controllers
    def dispatch(self, s, r, atomic=False, restart_ok=True):
        """Request that robot r takes over slot s.  atomic: immediate, reservation-checked.  Else queued (applied at end of step)."""
        if atomic:
            if r in self.taken() and s["exec"] != r: return False
            self._do_dispatch(s, r); return True
        self.queue.append((s, r)); return True

    def _flush_queue(self):
        used = set()
        for s, r in self.queue:
            if r in used or (r in self.taken() and s["exec"] != r):
                self.c["dup_dispatch"] += 1; s["lock"] = self.t + self.cfg.verify_delay; continue
            used.add(r); self._do_dispatch(s, r)
        self.queue = []

    def _do_dispatch(self, s, r):
        t = self.t; old = s["exec"]; c = self.c
        if old >= 0:
            c["reassign"] += 1; s["reass"] += 1
            c["necessary" if self.needs_replacement(old, s, t) else "unnecessary"] += 1
            if (not self.failed[old]) and (not self.infeasible_for(old, s, t)) and (not self.tele[old]) and not self.contained(s):
                c["dual_authority"] += 1; s["ho"] += self.cfg.conflict_penalty      # replacement collides with an unreachable, still-active robot
        else:
            c["fills"] += 1
        if self.infeasible_for(r, s, t):
            c["ineffective"] += 1
            key = ("fail", r) if self.failed[r] else (("snag", r) if self.snag[r] else (("restr", s["zone"], int(self.mode[r])) if self.restricted_now(s["zone"], self.mode[r], t) else ("cap", r)))
            if key in self.bad_causes: c["repeated_ineffective"] += 1
            self.bad_causes.add(key)
        if self.flagged(r): c["inv_flagged_dispatch"] += 1
        if self.age(r) > 4: c["inv_stale_dispatch"] += 1
        if old >= 0 and not self.ckpt_ok[r]:                                  # replacement cannot resume the checkpoint -> restart
            lost = s["prog"] * self.cfg.restart_loss; s["prog"] -= lost; c["work_lost"] += lost; c["restarts"] += 1
        s.update(exec=r, ho=s["ho"] + self.cfg.handover, rho=1.0, lease_last=t, sw=0, diag=None)
        s["rho_hist"].clear()

    def assign_initial(self, s, r):
        s.update(exec=r, ho=0, rho=1.0, lease_last=0)

    def start_diag(self, s, name):
        dur = DIAG_SPECS[name]; r = s["exec"]; t = self.t
        acc = self.rng2.random() < self.cfg.diag_acc
        if name == "self_test": truth = bool(self.snag[r])                       # supported = isolated robot fault
        else: truth = self.restricted_now(s["zone"], self.mode[r], t)             # supported = zone/mode restriction
        s["diag"] = dict(name=name, t_end=t + dur, truth=truth, conclusive=acc, r=r); self.c["diag_steps"] += dur; self.c["diag_runs"] += 1

    def switch_mode(self, s):
        r = s["exec"]; s["sw"] = 2; self.mode[r] = 1 - self.mode[r]; self.c["mode_switch"] += 1; s["rho_hist"].clear()

    # ------------------------------------------------------------------ execution
    def _execute(self, t):
        for s in self.slots:
            if s["done"] is not None: continue
            r = s["exec"]; prog = False
            if r >= 0:
                if s["ho"] > 0: s["ho"] -= 1
                elif s["sw"] > 0: s["sw"] -= 1
                elif s["diag"] is not None or t < s["esc_until"]: pass
                elif not self.infeasible_for(r, s, t) and not self.contained(s):
                    rate = (0.35 + 0.65 * self.u0[s["role"]][r]) * self.h[r]
                    if s["role"] == "rescuer":                                  # dependency: ground inspection trails accepted aerial survey
                        sc = self.slot_by[(s["tid"], 0)]
                        if sc["done"] is None: rate = min(rate, max(0.0, sc["prog"] - s["prog"]))
                    if rate > 1e-9: s["prog"] += rate; prog = True
                    self.c["util_sum"] += self.u0[s["role"]][r]; self.c["util_n"] += 1
            if prog:
                if s["idle"] > 0: self.incidents.append((s["idle"], s["idle_fault"])); s["idle"] = 0
            else:
                if s["idle"] == 0: s["idle_fault"] = (r < 0) or (r >= 0 and self.needs_replacement(r, s, t))
                s["idle"] += 1; self.c["downtime"] += 1
            self.c["slot_steps"] += 1
            if s["prog"] >= self.cfg.work:
                s["done"] = t; s["exec"] = -1
                if s["idle"] > 0: self.incidents.append((s["idle"], s["idle_fault"])); s["idle"] = 0

    def _progress(self):
        W = self.cfg.work; tot = ps = 0.0
        for tk in self.tasks:
            ss = [s for s in self.slots if s["tid"] == tk["id"]]
            ps += tk["prio"] * np.mean([min(1.0, s["prog"] / W) for s in ss]); tot += tk["prio"]
        return ps / tot

    def run(self):
        cfg = self.cfg; t_wall = time.perf_counter()
        for t in range(cfg.T):
            self.t = t
            self._apply_events(t); self.h = self._health(t); self._telemetry(t)
            for s in self.slots:                                                  # diagnostic completion
                d = s["diag"]
                if d is not None and t >= d["t_end"]: d["done"] = True
            if t == 0: self.ctrl.init(self)
            t0 = time.perf_counter(); self.ctrl.step(self, t); self.t_decide += time.perf_counter() - t0
            self._flush_queue(); self._execute(t)
            if t == cfg.deadline - 1: self.progress_deadline = self._progress()
        for s in self.slots:
            if s["idle"] > 0: self.incidents.append((s["idle"], s["idle_fault"]))
        return self._metrics(time.perf_counter() - t_wall)

    def _metrics(self, wall):
        cfg = self.cfg; c = self.c; done_t = []; comp = comp_dl = 0
        for tk in self.tasks:
            ss = [s for s in self.slots if s["tid"] == tk["id"]]
            if all(s["done"] is not None for s in ss):
                ft = max(s["done"] for s in ss); comp += 1; comp_dl += int(ft < cfg.deadline); done_t.append(ft)
            else: done_t.append(cfg.T)
        fault_inc = np.array([d for d, f in self.incidents if f]) if any(f for _, f in self.incidents) else np.array([])
        swap_inc = np.array([d for d, f in self.incidents if not f]) if any(not f for _, f in self.incidents) else np.array([])
        re = c["reassign"]; unrec = sum(1 for s in self.slots if s["done"] is None and (s["idle"] > 20))
        return dict(
            reassign=re, necessary=c["necessary"], unnecessary=c["unnecessary"], unnecessary_rate=(c["unnecessary"] / re) if re else np.nan,
            ineffective=c["ineffective"], repeated_ineffective=c["repeated_ineffective"], dup_dispatch=c["dup_dispatch"],
            dual_authority=c["dual_authority"], work_lost=c["work_lost"], restarts=c["restarts"], mode_switch=c["mode_switch"],
            escalations=c["escalations"], diag_steps=c["diag_steps"], diag_runs=c["diag_runs"],
            downtime=c["downtime"], availability=1 - c["downtime"] / max(1, c["slot_steps"]),
            recovery_latency=float(fault_inc.mean()) if len(fault_inc) else np.nan, recovery_p95=float(np.quantile(fault_inc, .95)) if len(fault_inc) else np.nan,
            swap_downtime=float(swap_inc.sum()) if len(swap_inc) else 0.0, unrecovered=unrec,
            progress_deadline=self.progress_deadline, progress_final=self._progress(), completion_rate=comp / len(self.tasks),
            on_time_rate=comp_dl / len(self.tasks), makespan=float(np.mean(done_t)),
            utility=(c["util_sum"] / c["util_n"]) if c["util_n"] else np.nan,
            inv_flagged=c["inv_flagged_dispatch"], inv_stale=c["inv_stale_dispatch"],
            decide_ms=1e3 * self.t_decide, wall_ms=1e3 * wall, decisions=self.n_decisions,
        )

## F. AhToVIK runtime controller
Interruption deadline (Eq. 4), ordered adaptation policy (Table I), evidence gate, authority leases, atomic reservation, verification, recovery cases.

In [ ]:
# ============================================================================
# F. AHTOVIK RUNTIME CONTROLLER (paper Sec. III-C..E)
#    * knowledge-guided monitoring: persistent deviations open an INTERRUPTION with a fixed deadline
#          t_end = min(t0 + B_T, d_T - r_T)  (Eq. 4), never reset when returning to analysis
#    * ordered adaptation policy (Table I): retain -> reconfigure incumbent -> admissible replacement
#          -> bounded observation -> escalate
#    * evidence gate: every prerequisite is SUPPORTED / CONTRADICTED / UNKNOWN; only fully supported candidates are admissible
#    * expiring execution authority: an unreachable robot is replaced only after its lease provably expired
#    * atomic reservation (generation-checked ownership), checkpoint-compatibility preference, verification interval H
#    * recovery cases + historical diagnostic selection (knowledge influences WHICH diagnostic is tried first, never admissibility)
# ============================================================================
@dataclass(frozen=True)
class AhDynCfg:
    persist: int = 4        # steps a deviation must persist before an interruption opens
    timeout: int = 6        # silent steps before a robot is declared unreachable
    theta: float = 0.5      # observed-progress ratio below which the role condition is violated
    BT: int = 30            # maximum automatic recovery budget B_T (steps)
    verify: int = 6         # verification interval H (steps after resumption)
    fresh: int = 4          # evidence freshness bound
    v: float = 0.5          # compromise weight
    use_gate: bool = True   # evidence gate + bounded interruption budget + escalation
    use_lease: bool = True  # expiring execution authority / containment before replacement
    use_diag: bool = True   # permitted diagnostics (bounded observation)
    reuse: bool = True      # historical knowledge may reorder diagnostics
    esc_cooldown: int = 25  # automatic retries stop for this long after an escalation (operator / policy takes over)

class Ahtovik(Controller):
    atomic = True; uses_kb = True
    def __init__(self, cfg=AhDynCfg(), name="AHTOVIK"):
        self.cfg, self.name = cfg, name; self.lease = cfg.use_lease

    # ------------------------------------------------------------------ team formation (stages 1-3 + gate)
    def _env_ok(self, w, s, c, t):
        return w.kb.status("restricted", (s["zone"], int(w.mode[c])), t) != SUPPORTED

    def form(self, w, t, initial=False):
        cfg = self.cfg
        for tk in sorted(w.tasks, key=lambda k: (-k["urgency"], k["deadline"], k["id"])):
            miss = [s for s in w.slots if s["tid"] == tk["id"] and s["exec"] < 0 and s["done"] is None and t >= s["esc_until"]]
            if not miss: continue
            counts = defaultdict(int); rep = {}
            for s in miss: counts[s["role"]] += 1; rep[s["role"]] = s
            ranked = {}
            for role, s in rep.items():
                idx = w.pool(s, fresh=cfg.fresh if cfg.use_gate else None)
                idx = np.array([c for c in idx if w.last_h[c] >= cfg.theta and (not cfg.use_gate or self._env_ok(w, s, c, t))], int)
                ranked[role] = w.q_rank(s, idx, cfg.v)
            if any(len(ranked[r]) < counts[r] for r in counts): continue          # no complete feasible team: the task stays pending
            order = sorted(counts, key=lambda r: len(ranked[r]) / counts[r])
            team, nodes, hit = dfs_team(order, counts, ranked, 20000)
            w.n_decisions += 1
            if team is None: continue
            used = defaultdict(int)
            for s in miss:
                r = team[s["role"]][used[s["role"]]]; used[s["role"]] += 1
                if initial: w.assign_initial(s, r)
                else: w.dispatch(s, r, atomic=True)
            w.kb.record_decision(dict(task=tk["id"], t=t, team=dict(team)))

    def init(self, w):
        for tk in w.tasks: tk.setdefault("urgency", tk["prio"])
        self.form(w, 0, initial=True)

    # ------------------------------------------------------------------ monitoring
    def step(self, w, t):
        if t % 5 == 0 and any(s["exec"] < 0 and s["done"] is None and t >= s["esc_until"] for s in w.slots): self.form(w, t)
        for s in w.slots:
            if s["done"] is not None or t < s["esc_until"] or s["exec"] < 0 or t < s["lock"]: continue
            if s["ho"] > 0 or s["sw"] > 0: continue
            st = s["st"]; intr = st.get("intr")
            d = s["diag"]
            if d is not None:
                if d.get("done"): self._finish_diag(w, s, t, intr)
                continue
            if intr is None:
                sym = self._detect(w, s, t)
                if sym is None: continue
                intr = self._open(w, s, t, sym)
            self._analyse(w, s, t, intr)

    def _detect(self, w, s, t):
        cfg = self.cfg; r = s["exec"]; st = s["st"]
        if w.flagged(r): return "hard"
        if w.age(r) >= cfg.timeout: return "unreachable"
        st["dev"] = st.get("dev", 0) + 1 if s["rho"] < cfg.theta else 0
        if st["dev"] >= cfg.persist: return "stall" if w.last_h[r] >= cfg.theta + 0.15 else "degraded"
        return None

    def _open(self, w, s, t, sym):
        cfg = self.cfg; r = s["exec"]; w.n_decisions += 1
        r_hat = (w.cfg.work - s["prog"]) / 0.6
        t_end = min(t + cfg.BT, s["deadline"] - r_hat) if cfg.use_gate else INF
        intr = dict(t0=t, t_end=t_end, sym=sym, m_inc=int(w.mode[r]), m0=int(w.mode[r]), r0=r, tried={r}, diags=[], state="analysis",
                    cause=None, action=None, good=0, bad=0, new=None, vstart=None)
        s["st"]["intr"] = intr; s["st"]["dev"] = 0
        return intr

    # ------------------------------------------------------------------ ordered adaptation policy (Table I)
    def _analyse(self, w, s, t, intr):
        cfg = self.cfg; r = s["exec"]
        if intr["state"] == "verifying": return self._verify(w, s, t, intr)
        if cfg.use_gate and t > intr["t_end"]: return self._escalate(w, s, t, intr)
        sym = intr["sym"]
        if sym == "unreachable":
            if w.age(r) < cfg.timeout: s["st"]["intr"] = None; return                    # telemetry returned: retain
            if cfg.use_lease and not w.authority_expired(s): return                         # authority may still be active: contain, do not replace yet
        elif sym in ("stall", "degraded") and len(s["rho_hist"]) >= 3 and min(list(s["rho_hist"])[-3:]) >= cfg.theta:
            s["st"]["intr"] = None; return                                                  # (1) retain: condition restored
        if cfg.use_gate and sym == "stall" and w.kb.status("restricted", (s["zone"], intr["m_inc"]), t) == SUPPORTED \
                and w.can_switch[r] and intr["action"] != "switch":                         # (2) reconfigure the incumbent in an approved alternative mode
            w.switch_mode(s); intr.update(state="verifying", action="switch", good=0, bad=0, vstart=None); return
        chosen, blocked = self._admissible(w, s, t, intr)                                    # (3) admissible replacement
        if chosen is not None:
            if w.dispatch(s, chosen, atomic=True):
                intr.update(state="verifying", action="replace", new=chosen, good=0, bad=0, vstart=None); intr["tried"].add(chosen)
            return
        if cfg.use_gate and cfg.use_diag and blocked:                                       # (4) bounded observation
            left = {d: du for d, du in DIAG_SPECS.items() if d not in intr["diags"] and du <= intr["t_end"] - t}
            if left:
                ctx = (s["role"], s["zone"], intr["m_inc"], "stall")
                order = w.kb.choose_diag(ctx, DIAG_SPECS, w.kb.episode) if cfg.reuse else sorted(DIAG_SPECS, key=lambda d: (DIAG_SPECS[d], d))
                name = next(d for d in order if d in left)
                w.start_diag(s, name); intr["diags"].append(name); return
        if cfg.use_gate: self._escalate(w, s, t, intr)                                      # (5) stop automatic retries, escalate

    def _admissible(self, w, s, t, intr):
        cfg = self.cfg
        idx = w.pool(s, fresh=cfg.fresh if cfg.use_gate else None, exclude=intr["tried"])
        if cfg.use_gate:
            idx = np.array([c for c in idx if w.kb.status("robot_fault", int(c), t) != SUPPORTED], int)
        ranked = w.q_rank(s, idx, cfg.v)
        if not cfg.use_gate:
            return (int(ranked[0]) if len(ranked) else None), False
        good, restart, blocked = [], [], False
        for c in ranked:
            if w.last_h[c] < cfg.theta + 0.1: continue                                      # capability prerequisite contradicted
            m = int(w.mode[c]); st_env = w.kb.status("restricted", (s["zone"], m), t)
            if st_env == SUPPORTED: continue                                                # environment prerequisite contradicted
            if st_env == UNKNOWN and intr["sym"] == "stall" and m == intr["m_inc"] and intr["cause"] is None:
                blocked = True; continue                                                    # environment prerequisite unknown -> not admissible
            (good if w.ckpt_ok[c] else restart).append(int(c))
        pick = good[0] if good else (restart[0] if restart else None)                       # prefer a candidate that can resume the checkpoint
        return pick, blocked

    # ------------------------------------------------------------------ diagnostics, verification, escalation
    def _finish_diag(self, w, s, t, intr):
        d = s["diag"]; s["diag"] = None
        if intr is None or not d["conclusive"]: return
        z, m, r = s["zone"], intr["m_inc"], d["r"]
        if d["name"] == "mode_check":
            if d["truth"]: w.kb.assert_("restricted", (z, m), SUPPORTED, t, ttl=90); intr["cause"] = "restriction"
            else: w.kb.assert_("restricted", (z, m), CONTRADICTED, t, ttl=40); intr["cause"] = "isolated"
        else:
            if d["truth"]:
                w.kb.assert_("robot_fault", int(r), SUPPORTED, t, ttl=200); w.kb.assert_("restricted", (z, m), CONTRADICTED, t, ttl=40)
                intr["cause"] = "isolated"

    def _verify(self, w, s, t, intr):
        cfg = self.cfg
        if s["ho"] > 0 or s["sw"] > 0: return
        if intr["vstart"] is None: intr["vstart"] = t
        if s["rho"] >= cfg.theta: intr["good"] += 1; intr["bad"] = 0
        else: intr["bad"] += 1
        if intr["good"] >= 3: return self._close(w, s, t, intr)
        if intr["bad"] >= 3 or t - intr["vstart"] >= cfg.verify:                           # unsuccessful attempt: record, return to analysis (same deadline)
            if intr["action"] == "replace" and intr["new"] is not None and cfg.use_gate:
                c = intr["new"]
                if int(w.mode[c]) == intr["m_inc"] and intr["sym"] == "stall":
                    w.kb.assert_("restricted", (s["zone"], intr["m_inc"]), SUPPORTED, t, ttl=90); intr["cause"] = "restriction"
                else: w.kb.assert_("robot_fault", int(c), SUPPORTED, t, ttl=200)
            if intr["action"] == "switch": intr["m_inc"] = int(w.mode[s["exec"]])
            intr.update(state="analysis", good=0, bad=0, vstart=None)

    def _close(self, w, s, t, intr):
        if intr["sym"] == "stall" and intr["cause"] in ("restriction", "isolated"):          # only VERIFIED outcomes with an established cause are retained
            ctx = (s["role"], s["zone"], intr["m0"], "stall")
            w.kb.add_case(ctx, intr["cause"], intr["diags"], intr["action"], t - intr["t0"], w.kb.episode)
        s["st"]["intr"] = None; s["st"]["dev"] = 0

    def _escalate(self, w, s, t, intr):
        s["esc_until"] = t + self.cfg.esc_cooldown; w.c["escalations"] += 1; s["st"]["intr"] = None; s["st"]["dev"] = 0

## G. Dynamic scenarios

In [ ]:
# ============================================================================
# G. DYNAMIC SCENARIOS (ground-truth fault schedules) and the paired runner
# ============================================================================
DYN_SCENARIOS = ["noise_only", "transient", "degrade", "hard", "silent", "restriction", "isolated", "stale_competing", "cascade", "mixed"]
SLOT_TARGETS = ["inc:0:0", "inc:0:1", "inc:0:2", "inc:1:0", "inc:1:1", "inc:2:0"]     # scout, 2 rescuers, 2 carriers, lifter

def dyn_events(kind, rng, k=None):
    U = lambda a, b: int(rng.integers(a, b + 1)); tgt = lambda: SLOT_TARGETS[int(rng.integers(0, len(SLOT_TARGETS)))]
    ev = []
    if kind == "noise_only": pass
    elif kind == "transient":
        for _ in range(U(2, 3)):
            if rng.random() < 0.5: ev.append(dict(kind="dip", t=U(10, 60), target=tgt(), dur=U(6, 14), level=float(rng.uniform(0.55, 0.70))))
            else: ev.append(dict(kind="dropout", t=U(10, 60), target=tgt(), dur=U(3, 14)))
    elif kind == "degrade":
        for _ in range(U(1, 2)): ev.append(dict(kind="ramp", t=U(10, 50), target=tgt(), length=10, final=0.2))
    elif kind == "hard":
        for _ in range(U(1, 2)): ev.append(dict(kind="hard", t=U(10, 60), target=tgt()))
    elif kind == "silent":
        for _ in range(U(1, 2)): ev.append(dict(kind="silent", t=U(10, 60), target=tgt()))
    elif kind == "restriction": ev.append(dict(kind="restrict", t=U(20, 40), target="inc:1:0", dur=60))
    elif kind == "isolated": ev.append(dict(kind="isolated", t=U(20, 40), target="inc:1:0"))
    elif kind == "stale_competing":
        t0 = U(20, 40)
        ev += [dict(kind="silent", t=t0, target="top_standby:1:0"), dict(kind="hard", t=t0 + 8, target="inc:1:0"), dict(kind="hard", t=t0 + 8, target="inc:2:0")]
    elif kind == "cascade":
        t0 = U(20, 40)
        ev += [dict(kind="hard", t=t0, target="inc:1:0"), dict(kind="hard", t=t0 + 6, target="inc:1:0"), dict(kind="hard", t=t0 + 14, target="inc:1:0")]
    elif kind == "mixed":
        pool = ["transient", "degrade", "hard", "silent", "restriction", "isolated", "stale_competing"]
        for kk in rng.choice(pool, size=(k or 3), replace=False): ev += dyn_events(str(kk), rng)
    elif kind == "mixed_k":
        pool = ["transient", "degrade", "hard", "silent", "restriction", "isolated"]
        for _ in range(k or 0): ev += dyn_events(pool[int(rng.integers(0, len(pool)))], rng)
    else: raise ValueError(kind)
    return ev

def make_dyn_instance(seed, cfg=None, **kw):
    cfg = cfg or DynCfg()
    return make_instance(cfg.n_robots, seed, templates=1, p_stale=0.0, p_restricted=0.0, p_unavail=0.0,
                         type_frac={"UAV": 0.2, "UGV": 0.6, "CREEPER": 0.2}, **kw)

def run_dyn(factory, scenario, seed, cfg=None, kb=None, k=None, events=None, inst=None, ctrl=None, attrs=None):
    """One paired run: fleet, fault schedule and telemetry noise are functions of (scenario, seed) only."""
    cfg = cfg or DynCfg()
    inst = inst or make_dyn_instance(seed, cfg)
    ev = events if events is not None else dyn_events(scenario, np.random.default_rng(MASTER_SEED + 5000 + seed * 13), k)
    ctrl = ctrl or factory()
    if kb is None and ctrl.uses_kb: kb = KnowledgeBase(reuse=getattr(ctrl.cfg, "reuse", True))
    w = World(inst, ev, cfg, ctrl, seed=MASTER_SEED + 9000 + seed, kb=kb, attrs=attrs)
    return w.run()

## H. Related-work baselines (dynamic)

In [ ]:
# ============================================================================
# H. DYNAMIC BASELINES (each is a compact, faithful-in-spirit re-implementation of the mechanism of a related-work approach,
#    running on the SAME plant, telemetry, checkpoint and handover substrate; none of them has an evidence gate, a diagnostic
#    layer, expiring authority or a bounded interruption budget unless stated).
#    MARKET     market-based immediate replacement (sequential auction on visible utility)          Gerkey & Mataric [13]
#    RAINBOW    architecture-based adaptation: ordered, design-time repair strategy with retries    Rainbow/Stitch [17][19], SUAVE/ReBeT [7][8]
#    MAYYA      capability estimation from observed performance + global optimal reallocation       Mayya et al. [5]
#    CALVO      online local repair + periodic global recomputation with hysteresis                 Calvo & Capitan [6]
#    FILIPPONE  trigger functions + alternative (backup) coalition per task                         Filippone et al. [12]
#    ESFAHANI   uncertainty-aware: act on posterior probability of failure, expected-utility choice Esfahani et al. [3]
#    CARWEHL    uncertainty reduction: add information acquisition (probe) when the posterior is ambiguous  Carwehl et al. [10]
#    CAMARA     latency-aware proactive adaptation from the trend of the observed capability        Camara et al. [11]
#    RODRIGUES  learning-based refinement: learns cause statistics online and ACTS on them          Rodrigues et al. [20]
# ============================================================================
def _phi(x):                                   # standard normal CDF without scipy overhead
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

class BaseCtl(Controller):
    """Shared mechanics: detection (explicit fault flag, silence timeout, persistent capability deviation) and replacement."""
    persist = 3; timeout = 6; theta = 0.55
    def init(self, w):
        for s in w.slots: s["st"]["dev"] = 0
        self.initial(w)

    def initial(self, w):                                                      # default initial team: utility-greedy in priority order
        taken = set()
        for s in w.slots:
            idx = np.array([i for i in w.pool(s) if i not in taken], int)
            if len(idx): r = int(idx[np.lexsort((w.ids[idx], -w.util_est(s, idx)))[0]]); w.assign_initial(s, r); taken.add(r)

    def detect(self, w, s, t):
        r = s["exec"]; st = s["st"]
        if w.flagged(r): return "hard"
        if w.age(r) >= self.timeout: return "silent"
        st["dev"] = st["dev"] + 1 if s["rho"] < self.theta else 0
        return "dev" if st["dev"] >= self.persist else None

    def candidates(self, w, s, exclude=()):
        idx = w.pool(s, exclude=exclude)
        return np.array([c for c in idx if w.last_h[c] >= 0.5], int)              # last-known health; freshness is NOT checked

    def score(self, w, s, idx): return w.util_est(s, idx)

    def pick(self, w, s, exclude=()):
        idx = self.candidates(w, s, exclude)
        return int(idx[np.lexsort((w.ids[idx], -self.score(w, s, idx)))[0]]) if len(idx) else None

    def active_slots(self, w, t):
        return [s for s in w.slots if s["done"] is None and s["exec"] >= 0 and t >= s["lock"] and s["ho"] == 0 and s["sw"] == 0]

    def fill_unstaffed(self, w, t):
        for s in w.slots:
            if s["done"] is None and s["exec"] < 0 and t >= s["lock"]:
                r = self.pick(w, s)
                if r is not None: w.dispatch(s, r, atomic=self.atomic)

    def step(self, w, t):
        if t % 5 == 0: self.fill_unstaffed(w, t)
        for s in self.active_slots(w, t):
            why = self.detect(w, s, t)
            if why is not None: w.n_decisions += 1; self.react(w, s, t, why)

    def react(self, w, s, t, why):
        r = self.pick(w, s, exclude=self.exclude(w, s))
        if r is not None: w.dispatch(s, r, atomic=self.atomic); s["st"]["dev"] = 0
    def exclude(self, w, s): return ()

# ---------------------------------------------------------------- MARKET: immediate replacement (auction on visible utility)
class Market(BaseCtl):
    name = "MARKET"; persist = 1; timeout = 3

# ---------------------------------------------------------------- RAINBOW / ReBeT: design-time ordered repair strategy
class Rainbow(BaseCtl):
    name = "RAINBOW"; persist = 3; timeout = 6; retries = 3
    def initial(self, w):
        super().initial(w)
        self.standby = {}
        for s in w.slots:                                        # strategy = ordered standby list fixed at design time (static utility ranking)
            idx = np.array([i for i in w.pool(s)], int)
            self.standby[(s["tid"], s["k"])] = [int(i) for i in idx[np.lexsort((w.ids[idx], -w.util_est(s, idx)))]]
    def react(self, w, s, t, why):
        st = s["st"]; n = st.get("tries", 0)
        if n >= self.retries: return                              # strategy exhausted: no further automatic repair
        r0 = s["exec"]
        if why == "dev" and w.last_h[r0] >= 0.65 and w.can_switch[r0] and not st.get("switched"):
            st["switched"] = True; st["tries"] = n + 1; w.switch_mode(s); st["dev"] = 0; return     # tactic 1: cheap blind repair (alternative mode)
        taken = w.taken()
        for r in self.standby[(s["tid"], s["k"])]:                # tactic 2: next standby of the design-time list
            if r not in taken and r != s["exec"] and not w.flagged(r) and r not in st.setdefault("used", set()):
                st["used"].add(r); st["tries"] = n + 1; w.dispatch(s, r, atomic=False); st["dev"] = 0; return

# ---------------------------------------------------------------- MAYYA: capability estimates + global optimal reallocation
class Mayya(BaseCtl):
    name = "MAYYA"; atomic = True; persist = 3; timeout = 6
    def initial(self, w):
        self.est = np.ones(w.n); self.global_assign(w, init=True)
    def step(self, w, t):
        for s in w.slots:                                          # capability estimate = EMA of observed progress ratio
            r = s["exec"]
            if r >= 0 and s["done"] is None and s["ho"] == 0 and w.tele[r]: self.est[r] = 0.7 * self.est[r] + 0.3 * max(0.0, s["rho"])
        super().step(w, t)
    def global_assign(self, w, init=False):
        slots = [s for s in w.slots if s["done"] is None]
        pool = [i for i in range(w.n) if not w.flagged(i) and self.est[i] > 0.5]
        if not slots or not pool: return
        val = np.full((len(slots), len(pool)), -1e6)
        for a, s in enumerate(slots):
            ok = w.E[s["role"]][pool]; u = w.util_est(s, np.array(pool)) * np.clip(self.est[pool], 0, 1)
            val[a] = np.where(ok, s["prio"] * u, -1e6)
        rows, cols = linear_sum_assignment(-val)
        for a, b in zip(rows, cols):
            s, r = slots[a], pool[b]
            if val[a, b] < 0 or s["exec"] == r: continue
            if init: w.assign_initial(s, r)
            else: w.dispatch(s, r, atomic=True)
    def react(self, w, s, t, why):
        st = s["st"]; st["dev"] = 0; self.est[s["exec"]] = min(self.est[s["exec"]], 0.2 if why != "dev" else self.est[s["exec"]])
        self.global_assign(w)

# ---------------------------------------------------------------- CALVO: local repair + periodic recomputation with hysteresis
class Calvo(BaseCtl):
    name = "CALVO"; atomic = True; persist = 3; timeout = 6; period = 25; gain = 0.10
    def step(self, w, t):
        super().step(w, t)
        if t > 0 and t % self.period == 0: self.recompute(w, t)
    def recompute(self, w, t):
        slots = [s for s in self.active_slots(w, t)]
        if not slots: return
        pool = [i for i in range(w.n) if not w.flagged(i) and w.last_h[i] >= 0.5]
        val = np.full((len(slots), len(pool)), -1e6)
        for a, s in enumerate(slots):
            val[a] = np.where(w.E[s["role"]][pool], s["prio"] * w.util_est(s, np.array(pool)), -1e6)
        rows, cols = linear_sum_assignment(-val)
        cur = sum(s["prio"] * w.util_est(s, np.array([s["exec"]]))[0] for s in slots)
        new = sum(val[a, b] for a, b in zip(rows, cols))
        if new > cur * (1 + self.gain):                             # accept the recomputed plan only if the gain justifies the churn
            for a, b in zip(rows, cols):
                if val[a, b] > 0 and slots[a]["exec"] != pool[b]: w.dispatch(slots[a], pool[b], atomic=True)

# ---------------------------------------------------------------- FILIPPONE: trigger functions + alternative coalition per task
class Filippone(BaseCtl):
    name = "FILIPPONE"; atomic = True; persist = 2; timeout = 4
    def initial(self, w):
        super().initial(w); self.backup = {}
        taken = set(w.taken())
        for tk in w.tasks:                                          # alternative sub-mission = disjoint backup coalition computed at planning time
            team = {}
            for s in [x for x in w.slots if x["tid"] == tk["id"]]:
                idx = np.array([i for i in w.pool(s) if i not in taken and i not in team.values()], int)
                if len(idx): r = int(idx[np.lexsort((w.ids[idx], -w.util_est(s, idx)))[0]]); team[(s["tid"], s["k"])] = r
            self.backup[tk["id"]] = team
    def react(self, w, s, t, why):
        team = self.backup.get(s["tid"], {}); slots = [x for x in w.slots if x["tid"] == s["tid"] and x["done"] is None]
        ok = len(team) == len(slots) and all(not w.flagged(r) and r not in w.taken() for r in team.values())
        if ok:                                                      # switch the whole task to its alternative coalition
            old = [x["exec"] for x in slots]
            for x in slots: w.dispatch(x, team[(x["tid"], x["k"])], atomic=True)
            self.backup[s["tid"]] = {}
            for x in slots: x["st"]["dev"] = 0
        else: super().react(w, s, t, why)

# ---------------------------------------------------------------- ESFAHANI: uncertainty-aware (posterior + expected utility)
class Esfahani(BaseCtl):
    name = "ESFAHANI"; persist = 1; timeout = 6; p_act = 0.80
    def p_fail(self, w, s):
        h = list(s["rho_hist"])[-4:]
        if not h: return 0.0
        sd = max(w.cfg.sigma, 1e-3) / math.sqrt(len(h)) + 0.02 * w.age(s["exec"])
        return _phi((self.theta - float(np.mean(h))) / sd)
    def detect(self, w, s, t):
        r = s["exec"]
        if w.flagged(r): return "hard"
        if w.age(r) >= self.timeout: return "silent"
        return "dev" if self.p_fail(w, s) > self.p_act else None
    def score(self, w, s, idx):                                     # expected utility: utility x P(feasible), staleness inflates uncertainty
        sd = np.maximum(w.cfg.sigma, 1e-3) + 0.03 * (w.t - w.last_t[idx])
        return w.util_est(s, idx) * np.array([_phi((w.last_h[i] - 0.5) / sd[k]) for k, i in enumerate(idx)])

# ---------------------------------------------------------------- CARWEHL: uncertainty reduction (information acquisition)
class Carwehl(Esfahani):
    name = "CARWEHL"; max_probes = 2; probe_cost = 3
    def detect(self, w, s, t):
        r = s["exec"]; st = s["st"]
        if w.flagged(r): return "hard"
        if w.age(r) >= self.timeout: return "silent"
        if t < st.get("probe_until", -1): return None
        p = self.p_fail(w, s)
        if 0.2 < p < 0.8 and st.get("probes", 0) < self.max_probes and len(s["rho_hist"]) >= 2:      # ambiguous: acquire information first
            st["probes"] = st.get("probes", 0) + 1; st["probe_until"] = t + self.probe_cost
            for _ in range(self.probe_cost): s["rho_hist"].append(float(w.h[r]) if not w.infeasible_for(r, s, t) else 0.0)
            w.c["probe_steps"] += self.probe_cost; return None
        return "dev" if p >= 0.8 else None

# ---------------------------------------------------------------- CAMARA: latency-aware proactive adaptation (trend extrapolation)
class Camara(BaseCtl):
    name = "CAMARA"; persist = 2; timeout = 5; margin = 4
    def detect(self, w, s, t):
        r = s["exec"]
        if w.flagged(r): return "hard"
        if w.age(r) >= self.timeout: return "silent"
        h = list(s["rho_hist"])
        if len(h) >= 4:
            slope = np.polyfit(np.arange(len(h)), h, 1)[0]
            if slope < -0.02:                                       # declining: replace early enough to cover the handover latency
                tt = (h[-1] - 0.5) / max(1e-6, -slope)
                if tt <= w.cfg.handover + self.margin: return "dev"
        st = s["st"]; st["dev"] = st["dev"] + 1 if s["rho"] < self.theta else 0
        return "dev" if st["dev"] >= self.persist else None

# ---------------------------------------------------------------- RODRIGUES: online learning of cause statistics, acts without diagnosis
class Rodrigues(BaseCtl):
    name = "RODRIGUES"; persist = 3; timeout = 6; p_act = 0.6; min_obs = 2
    def __init__(self): self.counts = defaultdict(lambda: [0, 0])      # context -> [#same-mode replacement failed, #succeeded]
    def react(self, w, s, t, why):
        st = s["st"]; r0 = s["exec"]; ctx = (s["zone"], int(w.mode[r0]))
        if why == "dev" and w.last_h[r0] >= 0.65:                          # stall with healthy self-diagnosis
            a, b = self.counts[ctx]
            skip_mode = (a + b >= self.min_obs) and a / (a + b) >= self.p_act      # learned belief: restriction -> avoid the mode directly
            st["pending"] = dict(ctx=ctx, t=t, mode=int(w.mode[r0]))
        else: skip_mode = False
        idx = self.candidates(w, s)
        if skip_mode: idx = np.array([c for c in idx if int(w.mode[c]) != ctx[1]], int)
        if len(idx):
            r = int(idx[np.lexsort((w.ids[idx], -self.score(w, s, idx)))[0]])
            if why == "dev": st["check"] = dict(r=r, ctx=ctx, t=t + w.cfg.handover + 8, same=int(w.mode[r]) == ctx[1])
            w.dispatch(s, r, atomic=False); st["dev"] = 0
    def step(self, w, t):
        for s in w.slots:                                              # outcome feedback after each replacement
            ck = s["st"].get("check")
            if ck and t >= ck["t"] and s["exec"] == ck["r"] and s["ho"] == 0 and ck["same"]:
                ok = s["rho"] >= self.theta; self.counts[ck["ctx"]][1 if ok else 0] += 1; s["st"]["check"] = None
        super().step(w, t)

# ---------------------------------------------------------------- factory registry
def ah(name, **kw): return lambda: Ahtovik(AhDynCfg(**kw), name)
DYN_METHODS = {
    "AHTOVIK":        ah("AHTOVIK"),
    "AHTOVIK-frozen": ah("AHTOVIK-frozen", reuse=False),
    "AHTOVIK-nogate": ah("AHTOVIK-nogate", use_gate=False, use_lease=False, use_diag=False),
    "MARKET": Market, "RAINBOW": Rainbow, "MAYYA": Mayya, "CALVO": Calvo, "FILIPPONE": Filippone,
    "ESFAHANI": Esfahani, "CARWEHL": Carwehl, "CAMARA": Camara, "RODRIGUES": Rodrigues,
}

## I. Sanity checks - all must PASS before trusting any result

In [ ]:
# ============================================================================
# K. SANITY CHECKS  (all must PASS before any result is trusted)
# ============================================================================
def sanity_checks():
    ok = lambda name, cond: print(("PASS  " if cond else "FAIL  ") + name) or bool(cond)
    res = []
    res.append(ok("AHP profiles consistent (CR < 0.1)", all(d["CR"] < 0.1 for d in ROLES.values())))
    # --- static
    inst = make_instance(60, 3)
    for name in ["AHTOVIK", "CO-MILP", "SAW-greedy", "SSI-auction"]:
        r = STATIC_METHODS[name](inst); a = [x for x in r["assign"] if x >= 0]
        valid = len(a) == len(set(a)) and all(inst.E[inst.slots[j][1]][x] for j, x in enumerate(r["assign"]) if x >= 0)
        res.append(ok(f"{name}: assignment respects eligibility and exclusivity", valid))
    gap_ok = True
    for sd in range(15):
        i2 = make_instance(60, 100 + sd); co = run_static("CO-MILP", i2)
        for m in ["SAW-greedy", "TOPSIS-greedy", "VIKOR-greedy", "TRAIT-greedy", "SSI-auction", "AHTOVIK-core"]:
            r = run_static(m, i2)
            if r["slots_filled"] == co["slots_filled"] and r["value"] > co["value"] + 1e-6: gap_ok = False
    res.append(ok("CO is a ceiling: no method with the same fill exceeds CO's value (15 instances)", gap_ok))
    ranked = {"a": np.array([0, 1, 2, 3]), "b": np.array([4, 5, 6, 7])}
    team, nodes, hit = dfs_team(["a", "b"], {"a": 1, "b": 2}, ranked, 1000, compat=lambda x, y: (x % 2) == (y % 2))
    res.append(ok("DFS team formation honours a cooperation constraint", team is not None and len({r % 2 for rs in team.values() for r in rs}) == 1))
    team, nodes, hit = dfs_team(["a"], {"a": 2}, {"a": np.array([0, 1])}, 1000, compat=lambda x, y: False)
    res.append(ok("DFS reports infeasibility instead of an invalid team", team is None))
    a1 = run_static("AHTOVIK", inst)["assigned_set"]; a2 = run_static("AHTOVIK", permute_instance(inst, np.random.default_rng(0).permutation(inst.N)))["assigned_set"]
    res.append(ok("AHTOVIK is order-independent (deterministic tie rule)", a1 == a2))
    # --- knowledge base
    kb = KnowledgeBase(); spec = DIAG_SPECS
    d0 = kb.choose_diag(("carrier", 1, 0, "stall"), spec, 0)
    for _ in range(3): kb.add_case(("carrier", 1, 0, "stall"), "restriction", ["self_test", "mode_check"], "switch", 10, 0)
    d1 = kb.choose_diag(("carrier", 1, 0, "stall"), spec, 1)
    res.append(ok("KB: default diagnostic order without history, historical order after recurring cases", d0[0] == "self_test" and d1[0] == "mode_check"))
    kb.new_episode(40); res.append(ok("KB: cases beyond the retention period are ignored", kb.choose_diag(("carrier", 1, 0, "stall"), spec, 40)[0] == "self_test"))
    # --- dynamic
    comp = all(run_dyn(DYN_METHODS[m], "noise_only", 1)["completion_rate"] == 1.0 for m in DYN_METHODS)
    res.append(ok("fault-free mission completes for every method", comp))
    a, b = run_dyn(DYN_METHODS["AHTOVIK"], "mixed", 5), run_dyn(DYN_METHODS["AHTOVIK"], "mixed", 5)
    res.append(ok("dynamic runs are deterministic for a fixed seed", all((a[k] == b[k]) or (isinstance(a[k], float) and np.isnan(a[k]) and np.isnan(b[k])) for k in a if k not in ("decide_ms", "wall_ms"))))
    viol = sum(run_dyn(DYN_METHODS["AHTOVIK"], sc, sd)[k] for sc in DYN_SCENARIOS for sd in range(6) for k in ("dual_authority", "dup_dispatch", "inv_flagged", "inv_stale"))
    res.append(ok("AHTOVIK: zero dual-authority / duplicate / flagged / stale dispatches over all scenarios", viol == 0))
    ng = sum(run_dyn(DYN_METHODS["AHTOVIK-nogate"], "mixed", sd)["repeated_ineffective"] for sd in range(10))
    res.append(ok(f"ablation sanity: removing the gate produces repeated ineffective interventions ({int(ng)} seen)", ng > 0))
    print("\nALL CHECKS PASSED" if all(res) else "\nSOME CHECKS FAILED - do not trust the results below")
    return all(res)

sanity_checks()

# Part 1 - Static decision benchmark
Latency, scalability, optimality, fill-rate, priority preservation, robustness, determinism.

In [ ]:
S = run_static_cases()
core, scar, scale, rv, rb, det = S["core"], S["scarcity"], S["scale"], S["robust_v"], S["robust_bigm"], S["determinism"]
RESULTS = {}                                                  # collected for the final scorecard

# ------------------------------------------------------------------ 1. LATENCY (N = 100)
lat = core.groupby("method").agg(build_ms=("build_ms", "median"), solve_ms=("solve_ms", "median"), total_ms=("total_ms", "median"),
                                 total_p95=("total_ms", lambda x: np.quantile(x, 0.95))).reindex(CORE_METHODS)
print(f"LATENCY, N={N_CORE}, {core.seed.nunique()} runs per method (median ms; solve_ms is the solver call for CO, the team-formation search for AHTOVIK)")
display(lat.round(3)) if "display" in globals() else print(lat.round(3))
lat.to_csv(f"{OUT}/static_latency.csv"); RESULTS["latency"] = lat

# ------------------------------------------------------------------ 2. SCALABILITY (log-log slope)
sc_med = scale.groupby(["method", "N"]).total_ms.median().unstack("N")
rows = []
for m in SCALE_METHODS:
    Ns = sc_med.columns.values; ts = sc_med.loc[m].values
    rows.append(dict(method=m, slope_all=loglog_slope(Ns, ts), slope_small=loglog_slope(Ns, ts, hi=300), slope_large=loglog_slope(Ns, ts, lo=1000),
                     ms_at_Nmin=ts[0], ms_at_Nmax=ts[-1], growth_x=ts[-1] / ts[0]))
slopes = pd.DataFrame(rows).set_index("method")
print(f"\nSCALABILITY: log-log slope of median total_time_ms vs N (N = {list(sc_med.columns)}; slope 1 = linear, <1 = fixed overhead dominates)")
display(slopes.round(3)) if "display" in globals() else print(slopes.round(3))
slopes.to_csv(f"{OUT}/static_scalability_slopes.csv"); sc_med.to_csv(f"{OUT}/static_scalability_medians.csv"); RESULTS["scalability"] = slopes
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for m in SCALE_METHODS:
    ax[0].loglog(sc_med.columns, sc_med.loc[m].values, marker="o", label=m, lw=2.6 if m.startswith("AHTOVIK") else 1.2)
    b = scale[scale.method == m].groupby("N")[["build_ms", "solve_ms"]].median()
ax[0].set_xlabel("N robots"); ax[0].set_ylabel("median total time (ms)"); ax[0].set_title("scalability"); ax[0].legend(fontsize=7)
for m, c in [("AHTOVIK", "C0"), ("CO-MILP", "C3")]:
    b = scale[scale.method == m].groupby("N")[["build_ms", "solve_ms"]].median()
    ax[1].loglog(b.index, b.build_ms, c + "-o", label=f"{m} build"); ax[1].loglog(b.index, b.solve_ms.clip(lower=1e-3), c + "--s", label=f"{m} solve")
ax[1].set_title("where the time lives: build vs solve"); ax[1].set_xlabel("N robots"); ax[1].legend(fontsize=8)
savefig("static_scalability")

# ------------------------------------------------------------------ 3. OPTIMALITY GAP vs the certified CO ceiling
ref = core[core.method == "CO-MILP"].set_index("seed")[["value"]].rename(columns={"value": "co_value"})
c = core.merge(ref, left_on="seed", right_index=True)
c["opt_gap"] = (c.co_value - c.value) / c.co_value
c["opt_gap_eff"] = (c.co_value - c.value_eff) / c.co_value            # same ceiling, but value realised by TRULY feasible robots only
c["opt_gap_pure"] = (c.co_pure_value - c.value) / c.co_pure_value      # ceiling of the pure-value objective (no BIG_M)
opt = mean_ci(c, ["method"], ["opt_gap", "opt_gap_eff", "opt_gap_pure"]).set_index("method").reindex(CORE_METHODS)
print(f"\nOPTIMALITY: gap = (CO_value - method_value)/CO_value, {c.seed.nunique()} instances (negative = method value above CO's BIG_M objective value)")
display(opt[["opt_gap", "opt_gap_lo", "opt_gap_hi", "opt_gap_eff", "opt_gap_pure"]].round(4)) if "display" in globals() else print(opt[["opt_gap", "opt_gap_lo", "opt_gap_hi", "opt_gap_eff", "opt_gap_pure"]].round(4))
opt.to_csv(f"{OUT}/static_optimality.csv"); RESULTS["optimality"] = opt
print("\nCO solver status codes (0 = optimal):", core[core.method == "CO-MILP"].status.value_counts().to_dict())

# ------------------------------------------------------------------ 4+5. FILL-RATE and PRIORITY-PRESERVATION under scarcity
fr = mean_ci(scar, ["method", "ratio"], ["fill_rate", "fill_rate_eff", "task_fill_rate_eff", "unmet_pw_norm", "unmet_pw_eff_norm"])
piv = lambda col: fr.pivot(index="method", columns="ratio", values=col).reindex([m for m in CORE_METHODS if m != "RANDOM"])
print("\nFEASIBILITY / FILL-RATE vs ground supply:demand ratio (fixed N = %d)" % N_SWEEP)
for col, title in [("fill_rate", "fill_rate = assigned/demanded"), ("fill_rate_eff", "fill_rate_eff (assigned robots that are truly feasible)"),
                   ("task_fill_rate_eff", "share of tasks with a COMPLETE, truly feasible team"),
                   ("unmet_pw_norm", "priority_weighted_unmet_demand / total priority demand (lower is better)"),
                   ("unmet_pw_eff_norm", "same, counting infeasible assignments as unmet")]:
    print("\n" + title); display(piv(col).round(3)) if "display" in globals() else print(piv(col).round(3))
fr.to_csv(f"{OUT}/static_fill_priority.csv"); RESULTS["fill"] = piv("fill_rate_eff"); RESULTS["unmet"] = piv("unmet_pw_norm"); RESULTS["unmet_eff"] = piv("unmet_pw_eff_norm")
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, col, ttl in zip(ax, ["fill_rate", "task_fill_rate_eff", "unmet_pw_norm"], ["fill-rate (assigned/demanded)", "complete & feasible tasks", "priority-weighted unmet demand (norm.)"]):
    for m in piv(col).index:
        a.plot(range(len(SCARCITY)), piv(col).loc[m].values, marker="o", label=m, lw=2.6 if m.startswith("AHTOVIK") else 1.1)
    a.set_xticks(range(len(SCARCITY))); a.set_xticklabels([f"{r}x" for r in SCARCITY]); a.set_xlabel("ground supply : demand"); a.set_title(ttl)
ax[0].legend(fontsize=7); savefig("static_scarcity")

# ------------------------------------------------------------------ 6. ROBUSTNESS / SENSITIVITY
rvt = mean_ci(rv, ["method", "v"], ["tau", "top1_same", "jaccard_vs_v05", "value"]).sort_values(["method", "v"])
print("\nROBUSTNESS (a): rank stability across the compromise weight v (reference v = 0.5) and team stability")
display(rvt[["method", "v", "tau", "top1_same", "jaccard_vs_v05", "value"]].round(3)) if "display" in globals() else print(rvt[["method", "v", "tau", "top1_same", "jaccard_vs_v05", "value"]].round(3))
rvt.to_csv(f"{OUT}/static_robust_v.csv"); RESULTS["robust_v"] = rvt
rbt = mean_ci(rb, ["big_m"], ["value", "fill", "jaccard_vs_1e6", "changed_vs_1e6"]).sort_values("big_m")
rbt["value_vs_pure_pct"] = rb.groupby("big_m").apply(lambda g: 100 * (g.value / g.pure_value).mean()).values
rbt["non_optimal_status_pct"] = rb.groupby("big_m").apply(lambda g: 100 * (g.status != 0).mean()).values
print("\nROBUSTNESS (b): CO solution stability across BIG_M (reference 1e6)")
display(rbt[["big_m", "value", "fill", "jaccard_vs_1e6", "changed_vs_1e6", "value_vs_pure_pct", "non_optimal_status_pct"]].round(4)) if "display" in globals() else print(rbt[["big_m", "value", "fill", "jaccard_vs_1e6", "changed_vs_1e6", "value_vs_pure_pct", "non_optimal_status_pct"]].round(4))
rbt.to_csv(f"{OUT}/static_robust_bigm.csv"); RESULTS["robust_bigm"] = rbt

# ------------------------------------------------------------------ 7. DETERMINISM / SOLUTION MULTIPLICITY
dt = det.groupby(["test", "method"]).agg(instances_changed_pct=("changed_any", lambda x: 100 * x.mean()),
                                         runs_changed_pct=("changed_frac", lambda x: 100 * x.mean())).reset_index()
print("\nDETERMINISM: % of instances in which the physical robot set changes (CO: tiny objective nudge x20; all methods: robot-array permutation x3)")
display(dt.round(2)) if "display" in globals() else print(dt.round(2))
dt.to_csv(f"{OUT}/static_determinism.csv"); RESULTS["determinism"] = dt

# Part 2 - Dynamic resilience benchmark

In [ ]:
# ============================================================================
# I. DYNAMIC RESILIENCE BENCHMARK RUNNERS
#    main     : every scenario x every method x N_DYN independent paired runs
#    sweeps   : resilience curves - fault intensity, telemetry noise, telemetry loss, spare-robot scarcity
#    sequences: knowledge-loop benchmark - N_SEQ independent sequences of recurring missions (see next cell)
# ============================================================================
ALL_DYN = list(DYN_METHODS)
SWEEP_METHODS = [m for m in ALL_DYN if m != "AHTOVIK-frozen"]
FAULT_K = [0, 1, 2, 3, 4, 6]
NOISE_LEVELS = [0.02, 0.05, 0.10, 0.20]
LOSS_LEVELS = [0.0, 0.05, 0.15]
SPARE_LEVELS = [28, 22, 18, 14]

def _dyn_pair(args):
    """All methods on one (scenario, seed) with identical fleet, fault schedule and telemetry noise."""
    scenario, seed, methods, cfg_kw, k, tag = args
    cfg = DynCfg(**cfg_kw); rows = []
    for m in methods:
        r = run_dyn(DYN_METHODS[m], scenario, seed, cfg=cfg, k=k)
        r.update(scenario=scenario, method=m, seed=seed, sweep=tag); rows.append(r)
    return rows

def run_main():
    items = [(sc, s, ALL_DYN, {}, None, "main") for sc in DYN_SCENARIOS for s in range(N_DYN)]
    return pd.DataFrame(flat(pmap(_dyn_pair, items, chunksize=4)))

def run_sweeps():
    items = []
    for k in FAULT_K:
        items += [("mixed_k", s, SWEEP_METHODS, {}, k, f"faults={k}") for s in range(N_DYN)]
    for sg in NOISE_LEVELS:
        items += [("mixed", s, SWEEP_METHODS, dict(sigma=sg), None, f"noise={sg}") for s in range(N_DYN)]
    for lp in LOSS_LEVELS:
        items += [("mixed", s, SWEEP_METHODS, dict(loss_prob=lp), None, f"loss={lp}") for s in range(N_DYN)]
    for nr in SPARE_LEVELS:
        items += [("mixed", s, SWEEP_METHODS, dict(n_robots=nr), None, f"robots={nr}") for s in range(N_DYN)]
    df = pd.DataFrame(flat(pmap(_dyn_pair, items, chunksize=4)))
    df["axis"] = df.sweep.str.split("=").str[0]; df["level"] = df.sweep.str.split("=").str[1].astype(float)
    return df

# ----------------------------------------------------------------------------- knowledge-loop (recurrence) benchmark
SEQ_STREAMS = ["recurring", "drift", "spurious", "nonrecurring"]
SEQ_METHODS = ["AHTOVIK", "AHTOVIK-frozen", "AHTOVIK-nogate", "RODRIGUES", "MARKET"]
N_EP = 8

def _seq_event(stream, ep, E, rng):
    t0 = int(rng.integers(20, 36)); target = "inc:1:0"
    if stream == "recurring": p = 0.9
    elif stream == "drift": p = 0.9 if ep < E // 2 else 0.0
    elif stream == "spurious": p = 1.0 if ep < 2 else 0.2
    else:
        p = 0.5; target = SLOT_TARGETS[int(rng.integers(0, len(SLOT_TARGETS)))]
    kind = "restrict" if rng.random() < p else "isolated"
    ev = dict(kind=kind, t=t0, target=target)
    if kind == "restrict": ev["dur"] = 70
    return [ev], kind

def _seq_one(args):
    stream, seq = args; cfg = DynCfg(); rows = []
    inst = make_dyn_instance(10_000 + seq, cfg)
    n = inst.N; attrs = dict(mode=np.zeros(n, int), can_switch=np.ones(n, bool))        # every robot has an approved alternative mode: recovery exists
    ctrls = {m: DYN_METHODS[m]() for m in SEQ_METHODS}                                   # one controller per sequence: learners keep their state
    kbs = {m: KnowledgeBase(reuse=getattr(ctrls[m].cfg, "reuse", True)) if ctrls[m].uses_kb else None for m in SEQ_METHODS}
    for ep in range(N_EP):
        rng = np.random.default_rng(MASTER_SEED + 77_000 + seq * 100 + ep)
        ev, kind = _seq_event(stream, ep, N_EP, rng)
        for m in SEQ_METHODS:
            if kbs[m] is not None: kbs[m].new_episode(ep)
            r = run_dyn(None, stream, 10_000 + seq * 10 + ep, cfg=cfg, events=ev, inst=inst, ctrl=ctrls[m], kb=kbs[m], attrs=attrs)
            r.update(stream=stream, method=m, seed=seq, episode=ep, cause=kind, kb_cases=(len(kbs[m].cases) if kbs[m] else 0)); rows.append(r)
    return rows

def run_sequences():
    return pd.DataFrame(flat(pmap(_seq_one, [(st, s) for st in SEQ_STREAMS for s in range(N_SEQ)], chunksize=2)))

def run_dynamic_cases():
    return dict(main=cached("dyn_main", run_main), sweeps=cached("dyn_sweeps", run_sweeps), seq=cached("dyn_seq", run_sequences))

In [ ]:
_trapz = getattr(np, 'trapezoid', None) or np.trapz

def a12(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    return (x[:, None] > y[None, :]).mean() + 0.5 * (x[:, None] == y[None, :]).mean()

def paired_p(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float); m = ~(np.isnan(a) | np.isnan(b)); a, b = a[m], b[m]
    if len(a) < 5 or np.allclose(a, b): return 1.0
    try: return float(stats.wilcoxon(a, b, zero_method="zsplit").pvalue)
    except Exception: return 1.0

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(o): run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def compare_to_ref(df, ref, metrics, lower_better):
    """Paired (per-seed) comparison of REF against every other method within each scenario; Holm within scenario."""
    rows = []
    for sc, g in df.groupby("scenario"):
        sub = []
        for m in metrics:
            P = g.pivot_table(index="seed", columns="method", values=m)
            for other in P.columns:
                if other == ref: continue
                ok = P[[ref, other]].dropna()
                if len(ok) < 5: continue
                sub.append(dict(scenario=sc, metric=m, other=other, n=len(ok), ref_mean=ok[ref].mean(), other_mean=ok[other].mean(),
                                p=paired_p(ok[ref], ok[other]), A12=a12(ok[ref], ok[other])))
        if sub:
            t = pd.DataFrame(sub); t["p_holm"] = holm(t.p.values); rows.append(t)
    res = pd.concat(rows, ignore_index=True)
    lb = res.metric.map(lambda m: lower_better[m])
    better = np.where(lb, res.ref_mean < res.other_mean, res.ref_mean > res.other_mean)
    res["verdict"] = np.where(res.p_holm >= 0.05, "n.s.", np.where(better, f"{ref} better", f"{ref} worse"))
    return res

D = run_dynamic_cases()
dm, dsw, dseq = D["main"], D["sweeps"], D["seq"]
dm["decide_ms_per_decision"] = dm.decide_ms / dm.decisions.replace(0, np.nan)

KEY = ["on_time_rate", "progress_deadline", "makespan", "downtime", "recovery_latency", "unnecessary", "repeated_ineffective",
       "dual_authority", "dup_dispatch", "work_lost", "escalations", "decide_ms"]
LOWER = {"on_time_rate": False, "progress_deadline": False, "makespan": True, "downtime": True, "recovery_latency": True, "unnecessary": True,
         "repeated_ineffective": True, "dual_authority": True, "dup_dispatch": True, "work_lost": True, "escalations": True, "decide_ms": True,
         "completion_rate": False, "ineffective": True, "reassign": True}

# ------------------------------------------------------------------ 1. mission continuity per scenario
print(f"DYNAMIC RESILIENCE BENCHMARK: {dm.seed.nunique()} independent paired runs per (scenario, method); {len(ALL_DYN)} methods")
pivots = {}
for m in KEY:
    pivots[m] = dm.pivot_table(index="scenario", columns="method", values=m, aggfunc="mean").reindex(DYN_SCENARIOS)[ALL_DYN]
    pivots[m].to_csv(f"{OUT}/dyn_pivot_{m}.csv")
for m in ["on_time_rate", "downtime", "recovery_latency", "repeated_ineffective", "dual_authority"]:
    print(f"\n{m}  (mean over runs)"); display(pivots[m].round(2)) if "display" in globals() else print(pivots[m].round(2))

# ------------------------------------------------------------------ 2. headline: macro average over scenarios + worst case
head = dm.groupby("method")[KEY + ["progress_final", "availability", "restarts", "diag_steps"]].mean()
per_sc = dm.groupby(["scenario", "method"]).on_time_rate.mean().unstack()
head["worst_case_on_time"] = per_sc.min().reindex(head.index)
head["decide_ms_per_decision"] = dm.groupby("method").decide_ms_per_decision.mean()
head = head.reindex(ALL_DYN).sort_values("on_time_rate", ascending=False)
print("\nHEADLINE (macro average over all scenarios; worst_case_on_time = lowest scenario mean)")
display(head[["on_time_rate", "worst_case_on_time", "progress_deadline", "makespan", "recovery_latency", "downtime", "unnecessary",
              "repeated_ineffective", "dual_authority", "dup_dispatch", "work_lost", "decide_ms_per_decision"]].round(3)) if "display" in globals() else print(head.round(3))
head.to_csv(f"{OUT}/dyn_headline.csv")

# ------------------------------------------------------------------ 3. paired statistics: AHTOVIK vs every baseline per scenario
cmp_metrics = ["progress_deadline", "downtime", "recovery_latency", "unnecessary", "repeated_ineffective", "dual_authority", "dup_dispatch"]
cmp = compare_to_ref(dm, "AHTOVIK", cmp_metrics, LOWER); cmp.to_csv(f"{OUT}/dyn_stats.csv", index=False)
summ = pd.crosstab([cmp.metric, cmp.other], cmp.verdict)
print("\nPAIRED TESTS (Wilcoxon, Holm within scenario, 5%): number of scenarios in which AHTOVIK is significantly better / worse / n.s. vs each method")
display(summ) if "display" in globals() else print(summ)
summ.to_csv(f"{OUT}/dyn_stats_summary.csv")

# ------------------------------------------------------------------ 4. figures: scenario heat map + per-scenario bars
fig, ax = plt.subplots(1, 3, figsize=(20, 5))
for a, m, ttl in zip(ax, ["on_time_rate", "downtime", "repeated_ineffective"], ["tasks completed before deadline", "interruption (slot-steps)", "repeated ineffective interventions"]):
    im = a.imshow(pivots[m].values, aspect="auto", cmap="viridis" if m == "on_time_rate" else "magma_r")
    a.set_xticks(range(len(ALL_DYN))); a.set_xticklabels(ALL_DYN, rotation=60, ha="right", fontsize=7); a.set_yticks(range(len(DYN_SCENARIOS))); a.set_yticklabels(DYN_SCENARIOS)
    a.set_title(ttl); plt.colorbar(im, ax=a, fraction=0.04)
    for i in range(pivots[m].shape[0]):
        for j in range(pivots[m].shape[1]): a.text(j, i, f"{pivots[m].values[i, j]:.1f}" if m != "on_time_rate" else f"{pivots[m].values[i, j]:.2f}", ha="center", va="center", fontsize=6, color="w")
    a.grid(False)
savefig("dyn_scenarios_heatmap")

# ------------------------------------------------------------------ 5. RESILIENCE CURVES (graceful degradation)
axes_def = [("faults", "number of injected faults"), ("noise", "telemetry noise sigma"), ("loss", "telemetry packet loss"), ("robots", "fleet size (spares)")]
res_rows = []
fig, ax = plt.subplots(2, 4, figsize=(21, 7.5))
for j, (axis, label) in enumerate(axes_def):
    sub = dsw[dsw.axis == axis]
    for i, metric in enumerate(["on_time_rate", "progress_deadline"]):
        for m in SWEEP_METHODS:
            g = sub[sub.method == m].groupby("level")[metric].mean()
            ax[i, j].plot(g.index, g.values, marker="o", ms=3, label=m, lw=2.8 if m == "AHTOVIK" else 1.0)
        ax[i, j].set_xlabel(label); ax[i, j].set_ylabel(metric)
    for m in SWEEP_METHODS:
        g = sub[sub.method == m].groupby("level").progress_deadline.mean(); lv = g.index.values.astype(float)
        sl = stats.linregress(lv, g.values).slope if len(lv) > 2 else np.nan
        auc = _trapz(g.values, lv) / (lv.max() - lv.min()) if len(lv) > 1 else np.nan
        res_rows.append(dict(axis=axis, method=m, auc_progress=auc, slope_progress=sl, worst=g.values.min(), at_hardest=g.values[-1 if axis != "robots" else 0]))
ax[0, 0].legend(fontsize=6, ncol=2); savefig("dyn_resilience_curves")
resil = pd.DataFrame(res_rows)
rs = resil.pivot(index="method", columns="axis", values="auc_progress")
rs["resilience_score"] = rs.mean(axis=1)
rs["worst_point"] = resil.groupby("method").worst.min()
rs = rs.sort_values("resilience_score", ascending=False)
print("\nRESILIENCE: normalised area under the progress-at-deadline curve (1.0 = no degradation) along each stress axis")
display(rs.round(4)) if "display" in globals() else print(rs.round(4))
rs.to_csv(f"{OUT}/dyn_resilience_scores.csv"); resil.to_csv(f"{OUT}/dyn_resilience_long.csv", index=False)
slopes_tbl = resil.pivot(index="method", columns="axis", values="slope_progress")
print("\nDEGRADATION SLOPE of progress-at-deadline per unit of stress (closer to 0 = more graceful)"); print(slopes_tbl.round(4))

# ------------------------------------------------------------------ 6. DECISION OVERHEAD (latency of the adaptation loop)
ovh = dm.groupby("method").agg(decide_ms_per_run=("decide_ms", "mean"), decisions=("decisions", "mean"), ms_per_decision=("decide_ms_per_decision", "mean"),
                               p95_decide_ms=("decide_ms", lambda x: np.quantile(x, .95))).reindex(ALL_DYN)
print("\nDECISION OVERHEAD per simulated mission (170 steps, 28 robots): wall time spent inside the controller"); display(ovh.round(3)) if "display" in globals() else print(ovh.round(3))
ovh.to_csv(f"{OUT}/dyn_overhead.csv")

# Part 3 - Knowledge-loop benchmark

In [ ]:
# ============================================================================
# J. KNOWLEDGE-LOOP BENCHMARK (does historical knowledge improve subsequent adaptation?)
#    Streams of recurring missions on a fixed fleet; every episode contains one interruption that can only be recovered after the
#    cause is established (zone/mode restriction -> reconfigure the incumbent, isolated fault -> replace it).
#    recurring   : restriction in 90% of the episodes      -> knowledge should help
#    drift       : restriction 90% for half of the stream, then never -> historical priors become wrong: harm must stay bounded
#    spurious    : first 2 episodes restriction, then 20%   -> early cases mislead
#    nonrecurring: random slot / zone / cause              -> contexts rarely match: no regression expected
# ============================================================================
dseq["diag_steps_pe"] = dseq.diag_steps
late = dseq[dseq.episode >= 2]
seq_tbl = late.groupby(["stream", "method"]).agg(recovery_latency=("recovery_latency", "mean"), downtime=("downtime", "mean"), diag_steps=("diag_steps", "mean"),
                                                   on_time=("on_time_rate", "mean"), unnecessary=("unnecessary", "mean"),
                                                   repeated_ineffective=("repeated_ineffective", "mean"), makespan=("makespan", "mean")).reset_index()
print(f"KNOWLEDGE LOOP: {dseq.seed.nunique()} independent sequences per stream, {N_EP} episodes each (episodes >= 2 shown)")
display(seq_tbl.round(3)) if "display" in globals() else print(seq_tbl.round(3))
seq_tbl.to_csv(f"{OUT}/kl_summary.csv", index=False)

# paired: frozen vs reuse (per-sequence means over episodes >= 2)
per_seq = late.groupby(["stream", "method", "seed"], as_index=False)[["recovery_latency", "downtime", "diag_steps", "makespan"]].mean()
rows = []
for st in SEQ_STREAMS:
    P = per_seq[per_seq.stream == st].pivot_table(index="seed", columns="method", values="recovery_latency")
    for other in ["AHTOVIK-frozen", "RODRIGUES", "MARKET", "AHTOVIK-nogate"]:
        ok = P[["AHTOVIK", other]].dropna()
        if len(ok) >= 5: rows.append(dict(stream=st, vs=other, ahtovik=ok.AHTOVIK.mean(), other=ok[other].mean(), delta=ok.AHTOVIK.mean() - ok[other].mean(),
                                          p=paired_p(ok.AHTOVIK, ok[other]), A12=a12(ok.AHTOVIK, ok[other])))
kcmp = pd.DataFrame(rows); kcmp["p_holm"] = holm(kcmp.p.values)
print("\nPaired comparison of RECOVERY LATENCY (AHTOVIK vs ...; delta < 0 = AHTOVIK recovers faster), Holm-corrected over this table")
display(kcmp.round(4)) if "display" in globals() else print(kcmp.round(4))
kcmp.to_csv(f"{OUT}/kl_stats.csv", index=False)

fig, ax = plt.subplots(2, 4, figsize=(20, 7), sharex=True)
for j, st in enumerate(SEQ_STREAMS):
    for i, (metric, ttl) in enumerate([("recovery_latency", "recovery latency (steps)"), ("diag_steps", "diagnostic steps spent")]):
        for m in SEQ_METHODS:
            g = dseq[(dseq.stream == st) & (dseq.method == m)].groupby("episode")[metric].mean()
            ax[i, j].plot(g.index, g.values, marker="o", ms=3, label=m, lw=2.8 if m == "AHTOVIK" else 1.1)
        ax[i, j].set_title(f"{st}: {ttl}"); ax[i, j].set_xlabel("episode")
ax[0, 0].legend(fontsize=7); savefig("kl_learning_curves")

# where does the time go: recovery latency by true cause, episodes >= 2
bycause = late.groupby(["stream", "cause", "method"]).recovery_latency.mean().unstack("method")[SEQ_METHODS]
print("\nRecovery latency by TRUE cause of the interruption (episodes >= 2): knowledge helps when the prior matches the cause, costs when it does not")
display(bycause.round(2)) if "display" in globals() else print(bycause.round(2))
bycause.to_csv(f"{OUT}/kl_by_cause.csv")
kb_growth = dseq[dseq.method == "AHTOVIK"].groupby(["stream", "episode"]).kb_cases.mean().unstack("stream")
print("\nVerified recovery cases accumulated in the knowledge base (mean per sequence):"); print(kb_growth.round(2))

# Implementation complexity and final scorecards

In [ ]:
# ============================================================================
# L. IMPLEMENTATION COMPLEXITY (measured from the source) + FINAL SCORECARDS
# ============================================================================
THIRD_PARTY = {"np": "numpy", "sparse": "scipy.sparse", "milp": "scipy.optimize (MILP / HiGHS)", "LinearConstraint": "scipy.optimize (MILP / HiGHS)",
               "Bounds": "scipy.optimize (MILP / HiGHS)", "linear_sum_assignment": "scipy.optimize (assignment)", "stats": "scipy.stats"}

def _source_of(o):
    """Source of a function/class. Classes defined in notebook cells may not expose their source -> fall back to their methods."""
    try: return textwrap.dedent(inspect.getsource(o))
    except (OSError, TypeError):
        if inspect.isclass(o):
            parts = []
            for v in o.__dict__.values():
                f = v.__func__ if isinstance(v, (staticmethod, classmethod)) else v
                if inspect.isfunction(f):
                    try: parts.append(textwrap.dedent(inspect.getsource(f)))
                    except (OSError, TypeError): pass
            attrs = [f"{k} = None" for k, v in o.__dict__.items() if not k.startswith("__") and not inspect.isfunction(v) and not isinstance(v, (staticmethod, classmethod))]
            return "\n".join(parts + attrs) if (parts or attrs) else None
        return None

def core_loc(objs):
    """Non-blank, non-comment, non-docstring lines of the given functions/classes (the method's core logic)."""
    total, deps = 0, set()
    for o in objs:
        src = _source_of(o)
        if src is None: return np.nan, set()
        tree = ast.parse(src); skip = set()
        for n in ast.walk(tree):
            if isinstance(n, (ast.FunctionDef, ast.ClassDef)) and n.body and isinstance(n.body[0], ast.Expr) and isinstance(getattr(n.body[0], "value", None), ast.Constant) and isinstance(n.body[0].value.value, str):
                skip |= set(range(n.body[0].lineno, n.body[0].end_lineno + 1))
            if isinstance(n, ast.Name) and n.id in THIRD_PARTY: deps.add(THIRD_PARTY[n.id])
        for i, line in enumerate(src.splitlines(), 1):
            if line.strip() and not line.strip().startswith("#") and i not in skip: total += 1
    return total, deps

_AH_CORE = [geometric_distances, compromise_index, dfs_team, _mm, KnowledgeBase]
COMPLEXITY_REGISTRY = {   # method -> (core objects, tunable parameters)
    # ---- static decision methods
    "AHTOVIK (static)": ([ahtovik_assign, AhtovikCfg] + _AH_CORE, ["v", "node_limit"]),
    "CO-MILP":          ([co_assign], ["big_m", "time_limit", "mip_rel_gap (solver)", "feasibility_tol (solver)"]),
    "SAW-greedy":       ([greedy_assign, sc_saw], []),
    "TOPSIS-greedy":    ([greedy_assign, sc_topsis, geometric_distances, _mm], []),
    "VIKOR-greedy":     ([greedy_assign, sc_vikor, _mm], ["v"]),
    "TRAIT-greedy":     ([greedy_assign, sc_trait], ["reference trait level"]),
    "SSI-auction":      ([ssi_assign], []),
    # ---- dynamic controllers (the runtime adaptation loop)
    "AHTOVIK (runtime)": ([Ahtovik, AhDynCfg, World.q_rank, World.dispatch, World.start_diag] + _AH_CORE,
                          ["persist", "timeout", "theta", "B_T", "verify H", "freshness", "v", "escalation cooldown", "lease", "clock margin", "k_min", "retention"]),
    "AHTOVIK-nogate":   ([Ahtovik, AhDynCfg, World.q_rank] + _AH_CORE[:3], ["persist", "timeout", "theta", "v"]),
    "MARKET":           ([BaseCtl, Market], ["persist", "timeout", "theta"]),
    "RAINBOW":          ([BaseCtl, Rainbow], ["persist", "timeout", "theta", "retries"]),
    "MAYYA":            ([BaseCtl, Mayya], ["persist", "timeout", "theta", "EMA factor", "estimate threshold"]),
    "CALVO":            ([BaseCtl, Calvo], ["persist", "timeout", "theta", "recompute period", "hysteresis gain"]),
    "FILIPPONE":        ([BaseCtl, Filippone], ["persist", "timeout", "theta"]),
    "ESFAHANI":         ([BaseCtl, Esfahani, _phi], ["timeout", "theta", "p_act", "posterior window", "staleness inflation"]),
    "CARWEHL":          ([BaseCtl, Esfahani, Carwehl, _phi], ["timeout", "theta", "p_act", "posterior window", "staleness inflation", "max probes", "probe cost", "ambiguity band (2)"]),
    "CAMARA":           ([BaseCtl, Camara], ["persist", "timeout", "theta", "margin", "slope threshold"]),
    "RODRIGUES":        ([BaseCtl, Rodrigues], ["persist", "timeout", "theta", "p_act", "min observations"]),
}
rows = []
for name, (objs, params) in COMPLEXITY_REGISTRY.items():
    loc, deps = core_loc(objs)
    rows.append(dict(method=name, lines_of_core_logic=loc, third_party_dependencies=len(deps), dependencies="; ".join(sorted(deps)), tunable_parameters=len(params)))
cplx = pd.DataFrame(rows)
print("IMPLEMENTATION COMPLEXITY (counted from this notebook's source; shared base-class lines are included for every controller that uses them)")
display(cplx) if "display" in globals() else print(cplx)
cplx.to_csv(f"{OUT}/complexity.csv", index=False)

# ------------------------------------------------------------------ STATIC SCORECARD (the factors of the evaluation table)
sm = ["AHTOVIK", "AHTOVIK-core", "CO-MILP", "SAW-greedy", "TOPSIS-greedy", "VIKOR-greedy", "TRAIT-greedy", "SSI-auction"]
sc = pd.DataFrame(index=sm)
sc["latency_ms (N=100)"] = RESULTS["latency"].total_ms
sc["build_ms"] = RESULTS["latency"].build_ms; sc["solve_ms"] = RESULTS["latency"].solve_ms
sc["scal_slope (all N)"] = RESULTS["scalability"].slope_all; sc["scal_slope (large N)"] = RESULTS["scalability"].slope_large
sc["ms at max N"] = RESULTS["scalability"].ms_at_Nmax
sc["opt_gap"] = RESULTS["optimality"].opt_gap; sc["opt_gap (feasible value)"] = RESULTS["optimality"].opt_gap_eff
sc["fill_eff @1.0x"] = RESULTS["fill"][1.0]; sc["fill_eff @0.8x"] = RESULTS["fill"][0.8]
sc["unmet_pw @0.8x"] = RESULTS["unmet"][0.8]; sc["unmet_pw_eff @0.8x"] = RESULTS["unmet_eff"][0.8]
rv_ = RESULTS["robust_v"].groupby("method").agg(tau=("tau", "mean"), jac=("jaccard_vs_v05", "mean"))
sc["rank stability tau (v sweep)"] = rv_.tau; sc["team stability Jaccard (v sweep)"] = rv_.jac
bm = RESULTS["robust_bigm"]; sc.loc["CO-MILP", "solution changed across BIG_M %"] = 100 * bm.changed_vs_1e6.mean()
dd = RESULTS["determinism"]
sc["order-dependence % (permutation)"] = dd[dd.test == "robot-order permutation"].set_index("method").instances_changed_pct
sc.loc["CO-MILP", "objective-nudge sensitivity %"] = dd[dd.test.str.startswith("CO objective")].instances_changed_pct.iloc[0]
sc["lines of core logic"] = cplx.set_index("method").lines_of_core_logic.reindex(["AHTOVIK (static)", "AHTOVIK (static)", "CO-MILP", "SAW-greedy", "TOPSIS-greedy", "VIKOR-greedy", "TRAIT-greedy", "SSI-auction"]).values
sc["tunable params"] = cplx.set_index("method").tunable_parameters.reindex(["AHTOVIK (static)", "AHTOVIK (static)", "CO-MILP", "SAW-greedy", "TOPSIS-greedy", "VIKOR-greedy", "TRAIT-greedy", "SSI-auction"]).values
print("\nSTATIC SCORECARD - every factor of the evaluation table, one row per decision method"); display(sc.round(3)) if "display" in globals() else print(sc.round(3))
sc.to_csv(f"{OUT}/scorecard_static.csv")

# ------------------------------------------------------------------ DYNAMIC SCORECARD (resilience factors)
dsc = head[["on_time_rate", "worst_case_on_time", "progress_deadline", "makespan", "recovery_latency", "downtime", "unnecessary",
            "repeated_ineffective", "dual_authority", "dup_dispatch", "work_lost", "decide_ms_per_decision"]].copy()
dsc["resilience score (AUC)"] = rs.resilience_score.reindex(dsc.index)
cmap = {"AHTOVIK": "AHTOVIK (runtime)", "AHTOVIK-frozen": "AHTOVIK (runtime)"}
dsc["lines of core logic"] = [cplx.set_index("method").lines_of_core_logic.get(cmap.get(m, m), np.nan) for m in dsc.index]
dsc["tunable params"] = [cplx.set_index("method").tunable_parameters.get(cmap.get(m, m), np.nan) for m in dsc.index]
fz = seq_tbl[seq_tbl.stream == "recurring"].set_index("method").recovery_latency
dsc["knowledge-loop: recovery latency (recurring)"] = fz.reindex(dsc.index)
print("\nDYNAMIC SCORECARD - resilience, continuity and loop cost, one row per method"); display(dsc.round(3)) if "display" in globals() else print(dsc.round(3))
dsc.to_csv(f"{OUT}/scorecard_dynamic.csv")
try:
    open(f"{OUT}/scorecard_static.tex", "w").write(sc.round(3).to_latex()); open(f"{OUT}/scorecard_dynamic.tex", "w").write(dsc.round(3).to_latex())
except Exception: pass

try:                                   # Colab: bundle and download
    import shutil; shutil.make_archive("ahtovik_validation_results", "zip", OUT)
    from google.colab import files; files.download("ahtovik_validation_results.zip")
except Exception as e:
    print("(not in Colab, or download skipped:", type(e).__name__, ")")

## Interpreting the output (checklist for the paper)
* **Latency / scalability:** report both the slope and where the time lives (build vs solve). AHTOVIK's cost is dominated by building the ranking; CO's by the solver call.
* **Optimality:** AHTOVIK is a compromise-ranking method, so a gap to CO is expected; report the gap together with the *feasible-value* gap and the effective fill-rate, where the evidence gate pays off.
* **Scarcity:** AHTOVIK staffs complete teams only; compare `fill_rate`, `task_fill_rate_eff` and priority-weighted unmet demand, and the `AHTOVIK-partial` variant.
* **Resilience:** look at worst-case scenarios and the degradation slopes, not only the average; report where AHTOVIK is slower (silent failures: lease wait; isolated faults: diagnostic wait; fault-free: Q-ranking is not utility-optimal).
* **Knowledge loop:** gains appear when the recorded cause matches the new one and turn into a small cost when it does not (drift / spurious streams).